In [3]:
# --- TOP-OF-NOTEBOOK GUARDS / PATHS BOOTSTRAP ---
import os, json, datetime as _dt

# Environment root
if os.path.exists("/kaggle/working"):
    BASE = "/kaggle/working"
elif os.path.exists("/content"):
    BASE = "/content"
else:
    BASE = "/mnt/data"
print("BASE =", BASE)
_p = lambda *p: os.path.join(BASE, *p)

# Single source of truth
CONFIG = {
    "RUN_PIPELINE": True,
    "RUN_UI": True,
    "RUN_MEDS": True,
    "RUN_SYNTH": False,
    "EQUIPMENT_STATUS_PATH": _p("equipment_status.csv"),
    "EQUIPMENT_MOVES_LOG_PATH": _p("moves_log.csv"),
    "SOP_REGISTRY_PATH": _p("sop_registry.csv"),
    "QR_OUTPUT_DIR": _p("qrs"),
    "EVENT_LOG_PATH": _p("event_log.jsonl"),
}

# FS prep
os.makedirs(CONFIG["QR_OUTPUT_DIR"], exist_ok=True)
os.makedirs(os.path.dirname(CONFIG["EVENT_LOG_PATH"]), exist_ok=True)

# Logger that always uses current CONFIG (no stale capture)
def _append_event(ev: dict):
    ev = {"ts": _dt.datetime.utcnow().isoformat()+"Z", **(ev or {})}
    with open(CONFIG["EVENT_LOG_PATH"], "a", encoding="utf-8") as f:
        f.write(json.dumps(ev, ensure_ascii=False) + "\n")

print("EVENT_LOG_PATH →", CONFIG["EVENT_LOG_PATH"])


BASE = /kaggle/working
EVENT_LOG_PATH → /kaggle/working/event_log.jsonl


In [4]:
# --- Load model bundle from attached dataset → set CONFIG path ---
import os, shutil
DATASET = "ed-pipeline-bundle-ui"  # ← replace with your actual dataset slug
SRC = f"/kaggle/input/{DATASET}/ed_phase2_model_thr_patched.joblib"
DST = "/kaggle/working/ed_phase2_model_thr_patched.joblib"

assert os.path.exists(SRC), f"Not found: {SRC} (check dataset slug/file name)"
if not os.path.exists(DST):
    os.makedirs(os.path.dirname(DST), exist_ok=True)
    shutil.copy2(SRC, DST)

CONFIG["MODEL_BUNDLE_PATH"] = DST
CONFIG["SKIP_MODEL_DISCOVERY"] = True  # prevents fallbacks from overriding this
print("Bundle ready →", CONFIG["MODEL_BUNDLE_PATH"])


Bundle ready → /kaggle/working/ed_phase2_model_thr_patched.joblib


In [5]:
CONFIG["RUN_SYNTH"] = True   # runs the realistic synthetic ML pipeline
CONFIG["RUN_MEDS"]  = True   # enables medication-plan parsing (paste-only if no OCR)
CONFIG["RUN_UI"]    = False  # keep off if widgets don't render in your Kaggle session


In [6]:

# CONFIG bootstrap with defaults per contract
from pathlib import Path
import os, json, pandas as pd

CONFIG = {
    "RUN_UI": False,
    "RUN_PIPELINE": False,
    "EQUIPMENT_STATUS_PATH": "/mnt/data/equipment_status.csv",
    "EQUIPMENT_MOVES_LOG_PATH": "/mnt/data/moves_log.csv",
    "SOP_REGISTRY_PATH": "/mnt/data/sop_registry.csv",
    "QR_OUTPUT_DIR": "/mnt/data/qrs",
    "EVENT_LOG_PATH": "/mnt/data/event_log.jsonl",
}

# Ensure paths exist with CSV/dir semantics (idempotent)
from pathlib import Path
from datetime import datetime, timezone
import csv
Path(CONFIG["QR_OUTPUT_DIR"]).mkdir(parents=True, exist_ok=True)
for f, header in [
    (CONFIG["EQUIPMENT_STATUS_PATH"], ["equipment_id","location","last_seen"]),
    (CONFIG["EQUIPMENT_MOVES_LOG_PATH"], ["timestamp","equipment_id","from","to"]),
    (CONFIG["SOP_REGISTRY_PATH"], ["id","title","url"]),
]:
    f = Path(f)
    if not f.exists():
        with f.open("w", newline="") as fp:
            csv.writer(fp).writerow(header)
# Seed SOP registry if empty
import os
if os.path.getsize(CONFIG["SOP_REGISTRY_PATH"]) < 40:
    with open(CONFIG["SOP_REGISTRY_PATH"], "a", newline="") as fp:
        csv.writer(fp).writerow(["sop-0001","Universal Precautions (offline)","about:blank"])
Path(CONFIG["EVENT_LOG_PATH"]).touch(exist_ok=True)


In [7]:

# --- Demo feature toggles (safe defaults) ---
CONFIG.setdefault("RUN_MEDS", False)
CONFIG.setdefault("MED_RULES_PATH", "/mnt/data/interaction_rules.json")
CONFIG.setdefault("ALLERGIES_PATH", "/mnt/data/patient_allergies.json")
CONFIG.setdefault("RUN_SYNTH", False)
CONFIG.setdefault("SAVE_SYNTH", False)


False

In [8]:

# -- Demo meds/ICU/ML flags (non-destructive update of CONFIG) --
CONFIG.setdefault("RUN_MEDS", False)
CONFIG.setdefault("RUN_SYNTH", False)
CONFIG.setdefault("SAVE_SYNTH", False)
CONFIG.setdefault("ALLERGIES_PATH", "/mnt/data/patient_allergies.json")
CONFIG.setdefault("MED_RULES_PATH", "/mnt/data/interaction_rules.json")


'/mnt/data/interaction_rules.json'

In [ ]:

# WorkflowState invariant: defined before use; exposes required methods; timers/backlogs/alerts intact
from dataclasses import dataclass, field
from typing import Any, Dict, List
import pandas as pd

@dataclass
class WorkflowState:
    role: str
    state: Dict[str, Any] = field(default_factory=dict)
    timers: Dict[str, float] = field(default_factory=dict)
    backlogs: Dict[str, List[Any]] = field(default_factory=dict)
    alerts: List[str] = field(default_factory=list)

    def touch_now(self, ts: pd.Timestamp):
        # update an example timer
        self.timers["last_touch_epoch"] = float(ts.value) / 1e9

    def feature_dict(self) -> Dict[str, Any]:
        # safe, extendable
        fd = {
            "role": self.role,
            "since_vitals_min": self.state.get("since_vitals_min", 0.0),
            "alerts_count": len(self.alerts),
        }
        # pass through any extra scalar features
        for k,v in self.state.items():
            if isinstance(v,(int,float,str)) and k not in fd:
                fd[k]=v
        return fd

    def update_state_from_event(self, event: Dict[str, Any]):
        # naive: merge event into state; track backlog
        self.state.update(event)
        self.backlogs.setdefault("events", []).append(event)

    def apply_event_log(self, events: List[Dict[str, Any]]):
        for ev in events:
            self.update_state_from_event(ev)


In [ ]:

# TinyCritics uses WorkflowState.feature_dict(); cold-start safe (no transform before fit)
import numpy as np

class TinyCritics:
    def __init__(self):
        self._fitted = False

    def fit(self, states, actions, rewards):
        # No-op fit to keep cold-start safe
        self._fitted = True
        return self

    def score(self, state: WorkflowState, actions: List[Dict[str,str]]):
        fd = state.feature_dict()
        n = len(actions)
        # Deterministic, bounded scores in [0,1]
        base = 0.5
        p = np.full(n, base, dtype=float)
        bonuses = np.zeros(n, dtype=float)
        uncertainty = np.full(n, 0.1, dtype=float)
        return p, bonuses, uncertainty


In [9]:

# Inline SOP surface and optional UI without external modules
from typing import Optional, Dict, Any
import csv, os
from pathlib import Path

def refresh_sop_registry(CONFIG: dict, base_url: Optional[str]) -> Dict[str, Any]:
    """
    Offline-safe: if base_url is provided and fetch works + CSV looks valid, overwrite the file.
    Otherwise, return a summary without raising. No sidecars.
    """
    path = Path(CONFIG["SOP_REGISTRY_PATH"])
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        with path.open("w", newline="") as fp:
            csv.writer(fp).writerow(["id","title","url"])
    if base_url:
        try:
            import requests
            resp = requests.get(base_url, timeout=5)
            resp.raise_for_status()
            text = resp.text.strip()
            rows = [r.split(",") for r in text.splitlines()]
            if rows and len(rows[0])>=3:
                with path.open("w", newline="") as fp:
                    csv.writer(fp).writerows(rows)
                return {"ok": True, "rows": len(rows)-1}
        except Exception as e:
            return {"ok": False, "error": str(e)}
    return {"ok": False, "error": "No base_url or unexpected format"}

# Optional, guarded UI demo (equipment status preview uses the CSV directly)
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W, pandas as pd
        eq_path = Path(CONFIG["EQUIPMENT_STATUS_PATH"])
        if not eq_path.exists():
            eq_path.write_text("equipment_id,location,last_seen\n")
        btn = W.Button(description="Show equipment status")
        out = W.Output()
        def _on_click(_):
            with out:
                out.clear_output()
                try:
                    df = pd.read_csv(eq_path)
                except Exception:
                    df = pd.DataFrame(columns=["equipment_id","location","last_seen"])
                display(df)
        btn.on_click(_on_click)
        display(W.VBox([btn, out]))
    except Exception as e:
        print("UI unavailable (optional):", e)


In [9]:
# === Self-contained ML→OPS bridge (no external file needed) ===
import os, glob, json
import numpy as np, pandas as pd
from joblib import load

# 1) Locate bundle
BASE = "/kaggle/working" if os.path.exists("/kaggle/working") else "/mnt/data"
BUNDLE = CONFIG.get("MODEL_BUNDLE_PATH")

if not (BUNDLE and os.path.exists(BUNDLE)):
    candidates = []
    for root in [BASE, "/mnt/data", "/content"]:
        candidates += glob.glob(os.path.join(root, "**", "ed_phase2_model*_patched.joblib"), recursive=True)
        candidates += glob.glob(os.path.join(root, "**", "*.joblib"), recursive=True)
        candidates += glob.glob(os.path.join(root, "**", "*.pkl"), recursive=True)
    BUNDLE = next((p for p in candidates if os.path.exists(p)), None)
    if not BUNDLE:
        raise FileNotFoundError("Model bundle not found. Set CONFIG['MODEL_BUNDLE_PATH'].")

# 2) Load and expose scorer
b = load(BUNDLE)
pipe = b.get("pipeline") or b.get("model")
cal  = b.get("calibrator")  # may be None (CalibratedClassifierCV wraps calibration)
THR  = float(b.get("threshold", 0.5))
FEAT = b.get("features")    # list or None

def score_proba(df: pd.DataFrame) -> np.ndarray:
    X = df[FEAT] if FEAT else df
    p = pipe.predict_proba(X)[:, 1]
    return cal.transform(np.asarray(p)) if cal is not None else p

def predict_one(row: dict) -> dict:
    p = float(score_proba(pd.DataFrame([row]))[0])
    return {"p": p, "y": int(p >= THR), "thr": THR}

CONFIG["MODEL_BUNDLE_PATH"] = BUNDLE
print("Loaded:", BUNDLE)
print("Threshold:", THR, "| Features:", len(FEAT) if FEAT else "infer from df")

# Optional: log load event if your _append_event exists
try:
    _append_event({"type":"model_loaded","bundle":BUNDLE,"thr":THR,"n_features":(len(FEAT) if FEAT else None)})
except Exception:
    pass


Loaded: /kaggle/working/ed_phase2_model_thr_patched.joblib
Threshold: 0.9988444286248084 | Features: 16


In [11]:
# After the bridge cell
def score_and_log(row: dict):
    """row must have the 16 feature keys in the bundle (FEAT)."""
    res = predict_one(row)  # uses THR/FEAT from the bridge
    try:
        _append_event({"type":"ml_score", "res": res, "keys": list(row.keys())})
    except Exception:
        pass
    print(f"ML risk p={res['p']:.3f} → {'ALERT' if res['y'] else 'ok'} (thr={res['thr']:.3f})")
    return res


In [12]:
from joblib import load
b = load(CONFIG["MODEL_BUNDLE_PATH"])
print("thr:", b["threshold"], "n_features:", len(b["features"]))
# Dummy row shape-check:
print(set(b["features"]) - set((FEAT or [])))  # should be empty


thr: 0.9988444286248084 n_features: 16
set()


In [ ]:

# === Inline ML → OPS bridge (no sidecars) ===
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, Any, Optional
import json, importlib.util, os

EVENT_LOG = Path(CONFIG["EVENT_LOG_PATH"])

def _append_event(ev: Dict[str, Any]):
    EVENT_LOG.parent.mkdir(parents=True, exist_ok=True)
    EVENT_LOG.touch(exist_ok=True)
    ev = {"ts": datetime.now(timezone.utc).isoformat(), **ev}
    with EVENT_LOG.open("a") as fp:
        fp.write(json.dumps(ev, ensure_ascii=False) + "\n")

def _detect_id_col(df):
    # Prefer meta.json hinted ID columns
    meta = Path("/mnt/data/meta.json")
    if meta.exists():
        try:
            js = json.loads(meta.read_text())
            for c in js.get("validated_id_cols", []):
                if c in df.columns:
                    return c
        except Exception:
            pass
    # common fallbacks
    for c in ["Fall-ID","fall_id","PatientID","patient_id","VISIT_ID","visit_id","ID","id"]:
        if c in df.columns:
            return c
    # else: first object-like
    for c in df.columns:
        if getattr(df[c], "dtype", None) == "object":
            return c
    return df.columns[0]

def run_user_pipeline(module_path: Optional[str]=None):
    """
    Execute the user's pipeline script in /mnt/data and return (test, probs_te, tau, id_col).
    Requires the script to expose globals: test, probs_te, tau.
    """
    cand = module_path or "/mnt/data/real_data_overlap_prevalence_isotonic(1).py"
    mp = Path(cand)
    if not mp.exists():
        mp = Path("/mnt/data/real_data_overlap_prevalence_isotonic.py")
    if not mp.exists():
        raise FileNotFoundError("Pipeline script not found.")
    spec = importlib.util.spec_from_file_location("user_pipeline_mod_inline", str(mp))
    mod = importlib.util.module_from_spec(spec)
    _cwd = os.getcwd()
    try:
        os.chdir("/mnt/data")
        spec.loader.exec_module(mod)  # type: ignore
    finally:
        os.chdir(_cwd)
    missing = [k for k in ["test","probs_te","tau"] if not hasattr(mod,k)]
    if missing:
        raise RuntimeError(f"Pipeline missing globals: {missing}")
    test = getattr(mod, "test")
    probs_te = getattr(mod, "probs_te")
    tau = float(getattr(mod, "tau"))
    id_col = _detect_id_col(test)
    return test, probs_te, tau, id_col, mp.name

def ml_to_ops_emit(test, probs_te, tau: float, id_col: str, source_name: str) -> Dict[str, Any]:
    n_events = 0
    pos = 0
    for i in range(len(test)):
        pid = test.iloc[i][id_col] if id_col in test.columns else i
        prob = float(probs_te[i])
        decision = "POS" if prob >= tau else "NEG"
        _append_event({
            "type": "ml_risk",
            "id_col": id_col,
            "patient_id": pid,
            "prob_cal": round(prob, 6),
            "tau": round(tau, 6),
            "decision": decision,
            "source": source_name,
        })
        n_events += 1
        if decision == "POS":
            pos += 1
            _append_event({
                "type": "lingering_alert",
                "patient_id": pid,
                "id_col": id_col,
                "prob_cal": round(prob, 6),
                "tau": round(tau, 6),
                "reason": "ml_high_risk",
                "source": "LingeringPatientMonitor"
            })
    _append_event({
        "type": "ml_risk_summary",
        "tau": round(tau, 6),
        "n_rows": int(len(test)),
        "id_col": id_col,
        "source": source_name
    })
    return {"n_events": n_events, "n_pos": pos, "tau": tau, "id_col": id_col}

# Guarded one-shot runner
if CONFIG.get("RUN_PIPELINE"):
    try:
        test, probs_te, tau, id_col, src = run_user_pipeline()
        res = ml_to_ops_emit(test, probs_te, tau, id_col, src)
        print("[OK] Emitted:", res)
        print("→ EVENT_LOG:", CONFIG["EVENT_LOG_PATH"])
    except Exception as e:
        print("[ERROR]", e)
else:
    print("Deferred… set CONFIG['RUN_PIPELINE']=True to run ML→OPS inline.")


In [ ]:

# Inline UI trigger (no sidecars)
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W
        btn = W.Button(description="Run ML → OPS (inline)", button_style="primary")
        out = W.Output()
        def _go(_):
            with out:
                out.clear_output()
                print("Running…")
                try:
                    test, probs_te, tau, id_col, src = run_user_pipeline()
                    res = ml_to_ops_emit(test, probs_te, tau, id_col, src)
                    print("[OK] Emitted:", res)
                except Exception as e:
                    print("[ERROR]", e)
        btn.on_click(_go)
        display(W.VBox([btn, out]))
    except Exception as e:
        print("UI unavailable:", e)
else:
    print("UI panel deferred… set CONFIG['RUN_UI']=True.")


In [ ]:
# ICU availability panel — replaces "actionable vs blocked" with explicit next-bed ETAs.
# No sidecars; all inline; guarded by RUN_UI.
from pathlib import Path
from datetime import datetime, timezone
import json

# Pre-seeded UKE ICUs (public info): names + capacities
UKE_UNITS = [
    {"name": "1A Neurochirurgische Intensivstation", "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1B Neurologische Intensivstation",    "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1C Interdisziplinäre Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1D Interdisziplinäre Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1E Interdisziplinäre Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1F Operative Intensivstation",        "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "1G Internistische Intensivstation",   "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "H1b Kardiologische Intensivstation",  "capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "H1b Kardiochirurgische Intensivstation","capacity": 12, "occupied": 12, "discharge_eta_minutes": []},
    {"name": "H2b Intensivstation Gefäß- und Herzmedizin", "capacity": 8, "occupied": 8, "discharge_eta_minutes": []},
]

def _format_eta(mins: int) -> str:
    if mins is None: return "unknown"
    if mins <= 0: return "now"
    h, m = divmod(int(mins), 60)
    return f"{m} min" if h == 0 else (f"{h} hr" if m == 0 else f"{h} hr {m} min")

def _load_icu_status(path="/mnt/data/icu_status.json"):
    p = Path(path)
    if p.exists():
        try:
            js = json.loads(p.read_text())
            if isinstance(js, dict) and js.get("units"):
                return js
        except Exception:
            pass
    # default to UKE units when nothing saved
    return {"units": list(UKE_UNITS), "timestamp": datetime.now(timezone.utc).isoformat()}

def _save_icu_status(js, path="/mnt/data/icu_status.json"):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(js, ensure_ascii=False, indent=2))

def _compute_next_bed_eta(unit):
    cap = int(unit.get("capacity", 0) or 0)
    occ = int(unit.get("occupied", 0) or 0)
    etas = [int(x) for x in (unit.get("discharge_eta_minutes") or []) if str(x).strip().isdigit()]
    if occ < cap: return 0
    return min(etas) if etas else None

if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W
        import pandas as pd

        state = _load_icu_status()
        units = state["units"]

        # UI widgets
        dd = W.Dropdown(options=[u.get("name","(unnamed)") for u in units] or ["(add a unit)"], description="Unit")
        name = W.Text(description="Name", placeholder="ICU-North")
        cap  = W.IntText(description="Capacity", value=12)
        occ  = W.IntText(description="Occupied", value=12)
        eta  = W.Text(description="ETAs (min)", placeholder="e.g. 30, 90, 180")

        add_btn = W.Button(description="Add/Update unit")
        calc_btn = W.Button(description="Estimate ETAs")
        save_btn = W.Button(description="Save status")
        out = W.Output()

        def _refresh_dropdown():
            dd.options = [u.get("name","(unnamed)") for u in units] or ["(add a unit)"]

        def _load_into_form(idx=0):
            if not units:
                name.value=""; cap.value=12; occ.value=12; eta.value=""; return
            u = units[idx]
            name.value = str(u.get("name",""))
            cap.value = int(u.get("capacity", 0) or 0)
            occ.value = int(u.get("occupied", 0) or 0)
            seq = u.get("discharge_eta_minutes") or []
            eta.value = ", ".join(str(int(x)) for x in seq)

        def _parse_eta(txt: str):
            out = []
            for chunk in txt.split(","):
                chunk = chunk.strip()
                if chunk:
                    try: out.append(int(float(chunk)))
                    except: pass
            return out

        def on_dd_change(change):
            if change["name"]=="value" and units:
                _load_into_form(dd.options.index(change["new"]))
        dd.observe(on_dd_change)

        def on_add(_):
            # no 'nonlocal' needed: we mutate the existing list
            u = {
                "name": name.value.strip() or f"ICU-{len(units)+1}",
                "capacity": int(cap.value or 0),
                "occupied": int(occ.value or 0),
                "discharge_eta_minutes": _parse_eta(eta.value),
            }
            names = [x.get("name","") for x in units]
            if u["name"] in names:
                units[names.index(u["name"])] = u
            else:
                units.append(u)
            _refresh_dropdown()
            dd.value = u["name"]
            with out:
                print(f"Saved unit '{u['name']}'")

        def on_calc(_):
            rows = []
            for u in units:
                eta_min = _compute_next_bed_eta(u)
                rows.append({
                    "ICU": u.get("name",""),
                    "capacity": int(u.get("capacity",0) or 0),
                    "occupied": int(u.get("occupied",0) or 0),
                    "next_bed_in": _format_eta(eta_min),
                })
            df = pd.DataFrame(rows) if rows else pd.DataFrame(columns=["ICU","capacity","occupied","next_bed_in"])
            with out:
                out.clear_output()
                if df.empty:
                    print("No units yet. Add a unit above.")
                else:
                    display(df.style.hide(axis='index'))
                    # Natural-language earliest
                    mins = [(r["ICU"], _compute_next_bed_eta(u)) for r,u in zip(rows, units)]
                    mins = [(n,m) for n,m in mins if m is not None]
                    if mins:
                        name_min, m = sorted(mins, key=lambda x: x[1])[0]
                        print(f"\nNext bed available on {name_min} in {_format_eta(m)}")
                    else:
                        print("\nNext bed availability: unknown (provide ETAs or reduce occupied < capacity).")

        def on_save(_):
            state["units"] = units
            state["timestamp"] = datetime.now(timezone.utc).isoformat()
            _save_icu_status(state)
            with out:
                print("Saved to /mnt/data/icu_status.json")

        add_btn.on_click(on_add)
        calc_btn.on_click(on_calc)
        save_btn.on_click(on_save)

        # Initial load
        _refresh_dropdown()
        if units:
            dd.value = dd.options[0]
            _load_into_form(0)

        display(W.VBox([
            W.HTML("<b>ICU next-bed availability</b>"),
            dd,
            W.HBox([name, cap, occ]),
            eta,
            W.HBox([add_btn, calc_btn, save_btn]),
            out
        ]))

    except Exception as e:
        print("ICU UI unavailable:", e)
else:
    print("ICU UI deferred… set CONFIG['RUN_UI']=True.")


In [ ]:

# === Realistic synthetic data pipeline (inline, no sidecars) ===
# Guard: RUN_SYNTH controls generation + training + event emission
from __future__ import annotations
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd, numpy as np, json
from typing import Dict, Any, Optional
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_curve, precision_recall_fscore_support, roc_auc_score

BASE = Path("/mnt/data")
EVENT_LOG = Path(CONFIG["EVENT_LOG_PATH"])

def _append_event(ev: Dict[str, Any]):
    EVENT_LOG.parent.mkdir(parents=True, exist_ok=True)
    EVENT_LOG.touch(exist_ok=True)
    ev = {"ts": datetime.now(timezone.utc).isoformat(), **ev}
    with EVENT_LOG.open("a") as fp:
        fp.write(json.dumps(ev, ensure_ascii=False) + "\n")

def _detect_label_and_id(train: pd.DataFrame, meta_path=BASE/"meta.json"):
    meta = {}
    if Path(meta_path).exists():
        try:
            meta = json.loads(Path(meta_path).read_text())
        except Exception:
            meta = {}
    label_col = meta.get("label_col")
    if not label_col:
        for c in train.columns:
            if pd.api.types.is_numeric_dtype(train[c]):
                u = set(pd.unique(train[c].dropna()))
                if u.issubset({0,1}):
                    label_col = c; break
    if not label_col:
        raise RuntimeError("Could not detect label column")
    id_col = None
    for c in meta.get("validated_id_cols", []):
        if c in train.columns: id_col = c; break
    for c in ["Fall-ID","fall_id","PatientID","patient_id","VISIT_ID","visit_id","ID","id"]:
        if id_col is None and c in train.columns: id_col = c; break
    if id_col is None: id_col = train.columns[0]
    return label_col, id_col

def _split_features(df: pd.DataFrame, id_col: str, label_col: str):
    num = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    cat = [c for c in df.columns if c not in num]
    drop = set([id_col, label_col])
    num = [c for c in num if c not in drop]
    cat = [c for c in cat if c not in drop]
    low_card = [c for c in cat if df[c].nunique(dropna=True) <= 30]
    return num, low_card

def _numeric_params(s: pd.Series):
    s_nonnull = s.dropna()
    if len(s_nonnull)==0:
        return {"mean":0.0,"std":1.0,"lo":0.0,"hi":1.0,"missing":1.0}
    mean = float(s_nonnull.mean()); std = float(s_nonnull.std(ddof=0) or 1.0)
    lo = float(np.percentile(s_nonnull, 1)); hi = float(np.percentile(s_nonnull, 99))
    missing = float(s.isna().mean())
    return {"mean":mean,"std":std,"lo":lo,"hi":hi,"missing":missing}

def _sample_numeric(n, p):
    x = np.random.normal(p["mean"], p["std"], size=n)
    x = np.clip(x, p["lo"], p["hi"])
    if p["missing"]>0:
        m = np.random.rand(n) < p["missing"]
        x = x.astype("float"); x[m] = np.nan
    return x

def _categorical_params(s: pd.Series):
    missing = float(s.isna().mean())
    counts = s.dropna().value_counts()
    if counts.empty:
        return {"cats":["UNK"],"probs":[1.0],"missing":1.0}
    cats = counts.index.tolist(); probs = (counts/counts.sum()).values.tolist()
    return {"cats":cats,"probs":probs,"missing":missing}

def _sample_categorical(n, p):
    base = np.random.choice(p["cats"], size=n, p=p["probs"])
    if p["missing"]>0:
        m = np.random.rand(n) < p["missing"]
        base = base.astype("object"); base[m] = None
    return base

def _prep_X(df: pd.DataFrame, num_cols, cat_cols, all_cols=None):
    X = pd.get_dummies(df[num_cols + cat_cols], columns=cat_cols, dummy_na=True)
    if all_cols is not None:
        for c in all_cols:
            if c not in X.columns: X[c] = 0
        X = X[all_cols]
    return X

def run_synth_pipeline(save_csv: bool=False):
    # Load real splits
    train = pd.read_csv(BASE/"train_DE_full.csv")
    val   = pd.read_csv(BASE/"val_DE_full.csv")
    test  = pd.read_csv(BASE/"test_DE_full.csv")

    label_col, id_col = _detect_label_and_id(train)
    num_cols, cat_cols = _split_features(train, id_col, label_col)

    # Fit a real model on real features (to induce structure for labels)
    Xr = _prep_X(train, num_cols, cat_cols); Xv = _prep_X(val, num_cols, cat_cols); Xt = _prep_X(test, num_cols, cat_cols)
    all_cols = sorted(set(Xr.columns) | set(Xv.columns) | set(Xt.columns))
    Xr = _prep_X(train, num_cols, cat_cols, all_cols); y_real = train[label_col].astype(int).values

    base_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("sc", StandardScaler(with_mean=False)),
                          ("lr", LogisticRegression(max_iter=1000))])
    base_pipe.fit(Xr, y_real)
    target_prev = float(np.mean(y_real))

    # Params from real TRAIN for synthesis
    num_param_map = {c: _numeric_params(train[c]) for c in num_cols}
    cat_param_map = {c: _categorical_params(train[c]) for c in cat_cols}

    def _make_ids(n): 
        import uuid
        return [f"SYN-{uuid.uuid4().hex[:10]}" for _ in range(n)]

    def synth_df(n_rows: int) -> pd.DataFrame:
        data = {id_col: _make_ids(n_rows)}
        for c in cat_cols: data[c] = _sample_categorical(n_rows, cat_param_map[c])
        for c in num_cols: data[c] = _sample_numeric(n_rows, num_param_map[c])
        return pd.DataFrame(data)

    n_tr, n_va, n_te = len(train), len(val), len(test)
    syn_tr = synth_df(n_tr); syn_va = synth_df(n_va); syn_te = synth_df(n_te)

    def label_from_base(df: pd.DataFrame) -> np.ndarray:
        X = _prep_X(df, num_cols, cat_cols, all_cols)
        scores = base_pipe.predict_proba(X)[:,1]
        thr = np.quantile(scores, 1-target_prev) if 0<target_prev<1 else 0.5
        return (scores >= thr).astype(int)

    for df in (syn_tr, syn_va, syn_te):
        df[label_col] = label_from_base(df)

    # Reorder
    def _reorder(df): 
        cols = [id_col, label_col] + [c for c in df.columns if c not in [id_col, label_col]]
        return df[cols]
    syn_tr, syn_va, syn_te = map(_reorder, (syn_tr, syn_va, syn_te))

    # Train calibrated model on SYNTH
    X_tr = _prep_X(syn_tr, num_cols, cat_cols); y_tr = syn_tr[label_col].astype(int).values
    X_va = _prep_X(syn_va, num_cols, cat_cols); y_va = syn_va[label_col].astype(int).values
    X_te = _prep_X(syn_te, num_cols, cat_cols); y_te = syn_te[label_col].astype(int).values
    all_synth_cols = sorted(set(X_tr.columns) | set(X_va.columns) | set(X_te.columns))
    X_tr = _prep_X(syn_tr, num_cols, cat_cols, all_synth_cols)
    X_va = _prep_X(syn_va, num_cols, cat_cols, all_synth_cols)
    X_te = _prep_X(syn_te, num_cols, cat_cols, all_synth_cols)

    base = Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("sc", StandardScaler(with_mean=False)),
                     ("lr", LogisticRegression(max_iter=1000))]).fit(X_tr, y_tr)
    cal = CalibratedClassifierCV(base, method="isotonic", cv="prefit").fit(X_va, y_va)

    probs_va = cal.predict_proba(X_va)[:,1]
    fpr, tpr, thr = roc_curve(y_va, probs_va)
    RECALL_FLOOR = 0.85
    meet = np.where(tpr >= RECALL_FLOOR)[0]
    if len(meet)>0:
        tau = float(thr[meet[0]])
    else:
        best_f1, best_t = -1, 0.5
        for t in np.linspace(0,1,201):
            yb = (probs_va >= t).astype(int)
            _, _, f1, _ = precision_recall_fscore_support(y_va, yb, average="binary", zero_division=0)
            if f1 > best_f1: best_f1, best_t = f1, t
        tau = float(best_t)

    probs_te = cal.predict_proba(X_te)[:,1]
    auc_te = float(roc_auc_score(y_te, probs_te))

    # Emit events (ml_risk_synth + lingering_alert_synth + summary)
    n_events = 0; n_pos = 0
    for i in range(len(syn_te)):
        pid = syn_te.iloc[i][id_col]
        p = float(probs_te[i])
        decision = "POS" if p >= tau else "NEG"
        ev = {"type":"ml_risk_synth","patient_id": pid,"id_col": id_col,
              "prob_cal": round(p,6), "tau": round(tau,6),"decision": decision,
              "source":"synth_pipeline"}
        _append_event(ev)
        n_events += 1
        if decision=="POS":
            n_pos += 1
            _append_event({"type":"lingering_alert_synth","patient_id": pid,"id_col": id_col,
                           "prob_cal": round(p,6), "tau": round(tau,6),
                           "reason":"ml_high_risk_synth","source":"LingeringPatientMonitor"})

    _append_event({"type":"ml_risk_summary_synth","tau": round(tau,6),"n_rows": int(len(syn_te)),
                   "id_col": id_col, "auc_te": round(auc_te,6), "source":"synth_pipeline"})

    # Optionally save CSVs
    if CONFIG.get("SAVE_SYNTH"):
        syn_tr.to_csv(BASE/"synth_train.csv", index=False)
        syn_va.to_csv(BASE/"synth_val.csv", index=False)
        syn_te.to_csv(BASE/"synth_test.csv", index=False)

    return {"n_events": n_events, "n_pos": n_pos, "tau": tau, "auc_te": auc_te, "id_col": id_col}

if CONFIG.get("RUN_SYNTH"):
    try:
        res = run_synth_pipeline(save_csv=bool(CONFIG.get("SAVE_SYNTH", False)))
        print("[SYNTH OK]", res)
        print("Events in:", CONFIG["EVENT_LOG_PATH"])
    except Exception as e:
        print("[SYNTH ERROR]", e)
else:
    print("Deferred… set CONFIG['RUN_SYNTH']=True to generate + run synthetic pipeline.")


In [ ]:

# Extend CONFIG for meds integration (idempotent)
CONFIG.update({
    "MED_RULES_PATH": "/mnt/data/interaction_rules.json",
    "ALLERGIES_PATH": "/mnt/data/patient_allergies.json",
    "RUN_MEDS": False,   # gate for meds OCR/parse pipeline
})
from pathlib import Path, PurePosixPath
# touch files if absent (empty defaults)
for p in [CONFIG["MED_RULES_PATH"], CONFIG["ALLERGIES_PATH"]]:
    pp = Path(p)
    if not pp.exists():
        try:
            # Minimal demo defaults
            if p.endswith("interaction_rules.json"):
                pp.write_text(json.dumps({
                    "pair_rules":[
                        # DEMO ONLY: not for clinical use. Replace with licensed source in production.
                        {"lhs":"sildenafil","rhs":"nitroglycerin","severity":"contraindicated","note":"Do not co-administer (demo)."},
                        {"lhs":"warfarin","rhs":"ibuprofen","severity":"major","note":"Bleeding risk (demo)."},
                        {"lhs":"ramipril","rhs":"spironolactone","severity":"major","note":"Hyperkalemia risk (demo)."},
                        {"lhs":"sertraline","rhs":"tramadol","severity":"moderate","note":"Serotonergic effects (demo)."},
                        {"lhs":"metformin","rhs":"iv contrast","severity":"moderate","note":"Renal function dependent (demo)."}
                    ],
                    "dup_atc_level": 5  # consider same 5th-level ATC a duplicate (demo)
                }, ensure_ascii=False, indent=2))
            elif p.endswith("patient_allergies.json"):
                pp.write_text(json.dumps({
                    # Example patient allergies (empty by default). Names free-text.
                    "allergies": []
                }, ensure_ascii=False, indent=2))
        except Exception:
            pass


In [ ]:

# === Medication Plan Import: OCR + Parse + Interaction/Allergy checks (inline, no sidecars) ===
from __future__ import annotations
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from datetime import datetime, timezone
import json, re

EVENT_LOG = Path(CONFIG["EVENT_LOG_PATH"])

def _append_event(ev: Dict[str, Any]):
    EVENT_LOG.parent.mkdir(parents=True, exist_ok=True)
    EVENT_LOG.touch(exist_ok=True)
    ev = {"ts": datetime.now(timezone.utc).isoformat(), **ev}
    with EVENT_LOG.open("a") as fp:
        fp.write(json.dumps(ev, ensure_ascii=False) + "\n")

# --- OCR (best-effort, optional) ---
def try_ocr(path: str) -> Optional[str]:
    """
    Try to OCR a BMP scan from an image/PDF.
    Returns UTF-8 text or None if OCR is unavailable.
    """
    try:
        from PIL import Image
        import pytesseract
        p = Path(path)
        if not p.exists():
            return None
        if p.suffix.lower() in {".png",".jpg",".jpeg",".tif",".tiff",".bmp"}:
            img = Image.open(p)
            txt = pytesseract.image_to_string(img, lang="deu")
            return txt
        elif p.suffix.lower() in {".pdf"}:
            # Lightweight PDF approach: try to use pdf2image if available, else None
            try:
                from pdf2image import convert_from_path
                pages = convert_from_path(str(p))
                chunks = []
                for im in pages[:5]:  # limit first 5 pages
                    chunks.append(pytesseract.image_to_string(im, lang="deu"))
                return "\n\n".join(chunks)
            except Exception:
                return None
        else:
            return None
    except Exception:
        return None

# --- BMP parser (heuristic) ---
_ATC_MAP = {
    # DEMO, extend as needed. Not for clinical use.
    "amoxicillin":"J01CA04","ibuprofen":"M01AE01","metoprolol":"C07AB02","ramipril":"C09AA05",
    "simvastatin":"C10AA01","acetylsalicylsäure":"B01AC06","aspirin":"B01AC06","warfarin":"B01AA03",
    "metformin":"A10BA02","insulin glargin":"A10AE04","insulin glargine":"A10AE04",
    "pantoprazol":"A02BC02","pantoprazole":"A02BC02","sertralin":"N06AB06","sertraline":"N06AB06",
    "spironolacton":"C03DA01","spironolactone":"C03DA01","sildenafil":"G04BE03",
    "nitroglycerin":"C01DA02","isosorbidmononitrat":"C01DA14","tramadol":"N02AX02"
}
_UNIT_PAT = r"(mg|mcg|µg|g|IE|I\.E\.|ml|mL)"
_DOSE_PAT = r"(?:(\d+(?:[.,]\d+)?)\s*"+_UNIT_PAT+r")"
_FREQ_PAT = r"\b(\d-\d-\d(?:-\d)?)\b|\b(morgens|mittags|abends|nachts)(?:[,/ ]+(morgens|mittags|abends|nachts))*\b"
_ROUTE_PAT = r"\b(p\.o\.|oral|i\.v\.|i\.m\.|s\.c\.|iv|im|sc|po|SL)\b"
_PRN_PAT = r"\bPRN\b|\bbei Bedarf\b"

def normalize(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip())

def parse_med_line(line: str) -> Optional[Dict[str, Any]]:
    s = line.strip()
    if not s: return None
    # quick reject of headers
    if re.search(r"(?i)medikationsplan|bundeseinheitlich|name des patienten|geburtsdatum|arzt|datum", s):
        return None
    # Split name + rest
    m = re.search(_DOSE_PAT, s, flags=re.I)
    name_part = s
    strength_val = None; strength_unit = None
    if m:
        name_part = s[:m.start()].strip(" -•:")
        strength_val = m.group(1).replace(",",".") if m.group(1) else None
        strength_unit = m.group(2)
    # freq
    freq = None
    mf = re.search(_FREQ_PAT, s, flags=re.I)
    if mf:
        freq = mf.group(0)
    # route
    mr = re.search(_ROUTE_PAT, s, flags=re.I)
    route = mr.group(0) if mr else None
    # prn
    prn = bool(re.search(_PRN_PAT, s, flags=re.I))

    name_clean = normalize(name_part.lower())
    if not name_clean:
        return None
    # take first token as candidate; remove extraneous punctuation
    base = re.sub(r"[^\wäöüß\- ]", "", name_clean).strip()
    # map to ATC where possible
    atc = None
    for k,v in _ATC_MAP.items():
        if k in base:
            atc = v; break
    return {
        "raw": s,
        "name": base,
        "strength": float(strength_val) if strength_val else None,
        "unit": strength_unit,
        "freq": freq,
        "route": route,
        "prn": prn,
        "atc": atc
    }

def parse_med_text(text: str) -> List[Dict[str, Any]]:
    meds = []
    # split on lines/bullets/semicolons
    for raw in re.split(r"[\n;\u2022]+", text):
        m = parse_med_line(raw)
        if m: meds.append(m)
    # de-dup by name+strength+unit
    uniq = {}
    for m in meds:
        key = (m["name"], m.get("strength"), m.get("unit"))
        if key not in uniq:
            uniq[key] = m
    return list(uniq.values())

# --- Interaction & allergy checking (pluggable rules) ---
def load_rules(path: str) -> Dict[str, Any]:
    try:
        js = json.loads(Path(path).read_text())
        return js if isinstance(js, dict) else {}
    except Exception:
        return {}

def load_allergies(path: str) -> Dict[str, Any]:
    try:
        js = json.loads(Path(path).read_text())
        if isinstance(js, dict) and "allergies" in js:
            return js
    except Exception:
        pass
    return {"allergies": []}

def atc_prefix(code: Optional[str], level=4) -> Optional[str]:
    if not code: return None
    code = code.strip().upper()
    # ATC format: A10BA02 -> levels 1:A, 2:A10, 3:A10B, 4:A10BA, 5:A10BA02
    if level==5: return code
    if level==4: return code[:5]
    if level==3: return code[:4]
    if level==2: return code[:3]
    if level==1: return code[:1]
    return code

def check_interactions(meds: List[Dict[str,Any]], rules: Dict[str,Any]) -> List[Dict[str,Any]]:
    warns = []
    # duplicate therapy via ATC 5th level
    dup_level = int(rules.get("dup_atc_level", 5))
    seen = {}
    for m in meds:
        pref = atc_prefix(m.get("atc"), dup_level)
        if pref:
            seen.setdefault(pref, []).append(m)
    for pref, group in seen.items():
        if len(group) > 1:
            warns.append({"type":"dup_therapy","atc_prefix":pref,"agents":[g.get("name") for g in group],
                          "severity":"info","note":f"Multiple agents in same ATC level {dup_level} (demo)"})
    # pairwise rules
    prules = rules.get("pair_rules", [])
    names = [m.get("name","") for m in meds]
    lower_names = [n.lower() for n in names]
    for r in prules:
        lhs = r.get("lhs","").lower()
        rhs = r.get("rhs","").lower()
        # match by name substring OR by ATC code exact if provided
        found_l = any(lhs in n for n in lower_names)
        found_r = any(rhs in n for n in lower_names)
        if found_l and found_r:
            warns.append({"type":"interaction_pair","lhs":lhs,"rhs":rhs,
                          "severity":r.get("severity","unknown"),"note":r.get("note","")})
    return warns

def check_allergies(meds: List[Dict[str,Any]], allergen_list: List[str]) -> List[Dict[str,Any]]:
    warns = []
    lowers = [a.lower() for a in allergen_list]
    for m in meds:
        nm = m.get("name","").lower()
        atc = (m.get("atc") or "").upper()
        if any(a in nm for a in lowers):
            warns.append({"type":"allergy_match","agent":m.get("name"),"severity":"major","note":"Listed allergy matches med name (string match)."})
        # soft cross-reactivity heuristic: same ATC first letter (very rough; DEMO ONLY)
        if atc and any(atc.startswith(a.upper()[:1]) for a in lowers if len(a)>=1):
            warns.append({"type":"allergy_crossreactivity_heuristic","agent":m.get("name"),
                          "severity":"info","note":"Heuristic ATC-class proximity; verify clinically."})
    return warns

def meds_pipeline_from_text(text: str, patient_id: Optional[str]=None) -> Dict[str,Any]:
    meds = parse_med_text(text)
    rules = load_rules(CONFIG["MED_RULES_PATH"])
    allergies = load_allergies(CONFIG["ALLERGIES_PATH"]).get("allergies", [])
    inter = check_interactions(meds, rules)
    alrx = check_allergies(meds, allergies)
    # Emit events
    _append_event({"type":"med_plan_import","n_meds": len(meds),"patient_id": patient_id})
    for m in meds:
        _append_event({"type":"med_entry","patient_id":patient_id, **m})
    for w in inter:
        _append_event({"type":"med_interaction_warning","patient_id":patient_id, **w})
    for w in alrx:
        _append_event({"type":"med_allergy_warning","patient_id":patient_id, **w})
    return {"meds": meds, "interactions": inter, "allergy_warnings": alrx}

def meds_pipeline_from_scan(path: str, patient_id: Optional[str]=None) -> Dict[str,Any]:
    txt = try_ocr(path)
    if not txt:
        return {"error":"OCR unavailable or failed; paste text instead."}
    return meds_pipeline_from_text(txt, patient_id=patient_id)

if CONFIG.get("RUN_MEDS"):
    print("Medication import pipeline is enabled. Use the UI panel (if RUN_UI) or call meds_pipeline_* functions.")
else:
    print("Deferred… set CONFIG['RUN_MEDS']=True to enable medication import pipeline.")


In [ ]:
# === Phase-2 calculators (extended, patched) + bundle + HL7 paste adapter ===
# Add this cell beneath your CONFIG/guards. No sidecars. Kaggle/Colab/Local safe.

from __future__ import annotations
from datetime import datetime, timezone
from typing import Dict, Any, Optional, List, Tuple
import json, re, math

# ---- Event sink wiring (uses your CONFIG if present) ----
try:
    EVENT_LOG_PATH = CONFIG.get("EVENT_LOG_PATH", "/mnt/data/event_log.jsonl")
except NameError:
    CONFIG = {"RUN_PIPELINE": True, "RUN_UI": True, "EVENT_LOG_PATH": "/mnt/data/event_log.jsonl"}
    EVENT_LOG_PATH = CONFIG["EVENT_LOG_PATH"]
    print("[guards] CONFIG was missing — seeded defaults:", CONFIG)

def _append_event(ev: Dict[str, Any]):
    ev = {"ts": datetime.utcnow().isoformat()+"Z", **ev}
    with open(EVENT_LOG_PATH, "a") as fp:
        fp.write(json.dumps(ev, ensure_ascii=False) + "\n")

# ---- helpers ----
def _pick(d: Dict[str, Any], names: List[str], cast=float, default=None):
    for n in names:
        if n in d and d[n] not in (None, ""):
            try: return cast(d[n])
            except Exception:
                try: return cast(str(d[n]).replace(",", "."))
                except Exception: pass
    return default
def _bool(v): 
    if isinstance(v, bool): return v
    s = str(v).strip().lower()
    return s in {"1","true","yes","y","ja","on","oui"}
def _safe_round(x, nd=1):
    try: return round(float(x), nd)
    except Exception: return x

# ---- calculators ----
def calc_qsofa(v: Dict[str,Any]): 
    rr=_pick(v,["rr","resp_rate"]); sbp=_pick(v,["sbp","systolic"]); gcs=_pick(v,["gcs"],float)
    avpu = (v.get("avpu") or "").upper()[:1]
    altered = (gcs is not None and gcs<15) or avpu in {"V","P","U"}
    return {"name":"qSOFA","score": int((rr is not None and rr>=22)) + int((sbp is not None and sbp<=100)) + int(altered)}

def calc_mews(v: Dict[str,Any]):
    def rr_s(x):  return 3 if x is not None and x<=8 else (0 if x and 9<=x<=14 else (1 if x and 15<=x<=20 else (2 if x and 21<=x<=29 else (3 if x and x>=30 else 0))))
    def hr_s(x):  return 2 if x is not None and x<=40 else (1 if x and 41<=x<=50 else (0 if x and 51<=x<=100 else (1 if x and 101<=x<=110 else (2 if x and 111<=x<=129 else (3 if x and x>=130 else 0)))))
    def sbp_s(x): return 3 if x is not None and x<=70 else (2 if x and 71<=x<=80 else (1 if x and 81<=x<=100 else (0 if x and 101<=x<=199 else (2 if x and x>=200 else 0))))
    def t_s(x):   return 2 if x is not None and x<=35.0 else (1 if x and 35.1<=x<=36.0 else (0 if x and 36.1<=x<=38.0 else (1 if x and 38.1<=x<=38.5 else (2 if x and x>=38.6 else 0))))
    def avpu_s(x): return {"A":0,"V":1,"P":2,"U":3}.get((x or "A").upper()[:1],0)
    return {"name":"MEWS","score": int(rr_s(_pick(v,["rr"]))+hr_s(_pick(v,["hr","pulse"]))+sbp_s(_pick(v,["sbp"]))+t_s(_pick(v,["temp"]))+avpu_s(v.get("avpu")))}

def calc_heart(p: Dict[str,Any]):
    age=_pick(p,["age"],int); hist=_pick(p,["heart_history"],int); ecg=_pick(p,["heart_ecg"],int); risk=_pick(p,["heart_risk"],int)
    trop=_pick(p,["troponin","hs_troponin","trop"]); uln=_pick(p,["troponin_uln"],float)
    age_s = 2 if (age is not None and age>=65) else (1 if (age is not None and 45<=age<=64) else 0)
    ratio = (trop/uln) if (trop is not None and uln) else None
    trop_s = 2 if (ratio is not None and ratio>3) else (1 if (ratio is not None and 1<ratio<=3) else (0 if ratio is not None else 0))
    total = (hist or 0)+(ecg or 0)+age_s+(risk or 0)+trop_s
    return {"name":"HEART","score": int(total)}

def calc_grace_coarse(p: Dict[str,Any]):
    age=_pick(p,["age"],int); hr=_pick(p,["hr","pulse"]); sbp=_pick(p,["sbp"]); crea=_pick(p,["creatinine"])
    sc=0
    if age is not None: sc += (0 if age<40 else 20 if age<60 else 40 if age<80 else 60)
    if hr  is not None: sc += (0 if hr<70  else 10 if hr<90  else 20 if hr<110 else 30 if hr<150 else 40)
    if sbp is not None: sc += (40 if sbp<80 else 30 if sbp<100 else 10 if sbp<120 else 0)
    if crea is not None: sc += (0 if crea<1.2 else 10 if crea<2.0 else 20 if crea<3.0 else 30)
    return {"name":"GRACE_coarse","score": int(sc)}

def calc_sofa_min(v: Dict[str,Any], labs: Dict[str,Any]):
    pf=None; pao2=_pick(labs,["pao2"]); fio2=_pick(labs,["fio2"])
    if pao2 is not None and fio2: 
        try: pf=float(pao2)/float(fio2)
        except Exception: pf=None
    plate=_pick(labs,["platelets","plt"]); bili=_pick(labs,["bilirubin"]); mapv=_pick(v,["map"]); gcs=_pick(v,["gcs"]); crea=_pick(labs,["creatinine"])
    sc=0
    if pf is not None:    sc += (4 if pf<100 else 3 if pf<200 else 2 if pf<300 else 1 if pf<400 else 0)
    if plate is not None: sc += (4 if plate<20 else 3 if plate<50 else 2 if plate<100 else 1 if plate<150 else 0)
    if bili  is not None: sc += (4 if bili>=12 else 3 if bili>=6 else 2 if bili>=2 else 1 if bili>=1.2 else 0)
    if mapv  is not None: sc += (1 if mapv<70 else 0)
    if gcs   is not None: sc += (4 if gcs<6 else 3 if gcs<10 else 2 if gcs<13 else 1 if gcs<15 else 0)
    if crea  is not None: sc += (4 if crea>=5 else 3 if crea>=3.5 else 2 if crea>=2 else 1 if crea>=1.2 else 0)
    return {"name":"SOFA_min","score": int(sc)}

def assess_d_dimer(value, unit, age, pregnant=False):
    if value is None: return {"available": False}
    unit=(unit or "").lower()
    val_ug = float(value)*1000.0 if "mg/l" in unit else float(value)  # assume μg/L FEU default
    thr = 500.0
    if age is not None and age>50 and not pregnant: thr = float(age)*10.0
    return {"available": True, "value_ug_per_l": val_ug, "thr_ug_per_l": thr, "ok_below_thr": bool(val_ug < thr)}

def assess_troponin_delta(series: List[Tuple[Optional[datetime], float, str]]):
    if not series or len(series)<2: return {"available": False}
    try: series = sorted(series, key=lambda x: (x[0] or datetime.min))
    except Exception: pass
    (t0,v0,u0),(t1,v1,u1)=series[-2],series[-1]
    def to_ng_l(v,u):
        u=(u or "").lower()
        if "ng/l" in u or "pg/ml" in u: return float(v)
        if "µg/l" in u or "ug/l" in u:  return float(v)*1000.0
        return float(v)
    p=to_ng_l(v0,u0); c=to_ng_l(v1,u1); d=c-p; pct=(abs(d)/p*100.0) if p!=0 else None
    flag = (abs(d)>=51.0) or (pct is not None and pct>=20.0)
    return {"available": True, "prev": p, "curr": c, "delta_abs": d, "delta_pct": pct, "flag": bool(flag)}

def calc_wells_pe(p: Dict[str,Any]):
    pts=0.0
    pts+=3.0 if _bool(p.get("dvt_signs")) else 0.0
    pts+=3.0 if _bool(p.get("pe_most_likely")) else 0.0
    pts+=1.5 if ((_pick(p,["hr","pulse"],float) or 0)>100) else 0.0
    pts+=1.5 if (_bool(p.get("immobilized")) or _bool(p.get("recent_surgery_4w"))) else 0.0
    pts+=1.5 if _bool(p.get("prev_vte")) else 0.0
    pts+=1.0 if _bool(p.get("hemoptysis")) else 0.0
    pts+=1.0 if _bool(p.get("cancer_active")) else 0.0
    return {"name":"WELLS_PE","score": pts, "tier2": ("likely" if pts>4 else "unlikely")}

def calc_wells_dvt(p: Dict[str,Any]):
    pts=0
    pts+=1 if _bool(p.get("cancer_active")) else 0
    pts+=1 if (_bool(p.get("paresis")) or _bool(p.get("plaster_cast"))) else 0
    pts+=1 if (_bool(p.get("bedridden_3d")) or _bool(p.get("surgery_12w"))) else 0
    pts+=1 if _bool(p.get("deep_vein_tenderness")) else 0
    pts+=1 if _bool(p.get("entire_leg_swollen")) else 0
    pts+=1 if _bool(p.get("calf_swelling_gt3cm")) else 0
    pts+=1 if _bool(p.get("pitting_edema")) else 0
    pts+=1 if _bool(p.get("collateral_nonvaricose")) else 0
    pts+=1 if _bool(p.get("prev_dvt")) else 0
    pts-=2 if _bool(p.get("alt_dx_as_likely")) else 0
    return {"name":"WELLS_DVT","score": int(pts), "tier2": ("likely" if pts>=2 else "unlikely")}

def calc_perc(p: Dict[str,Any]):
    crit = {
        "age<50": int((_pick(p,["age"],int) or 10) < 50),
        "hr<100": int((_pick(p,["hr","pulse"],float) or 0) < 100),
        "sao2>=95": int((_pick(p,["sao2","spo2"],float) or 0) >= 95),
        "no_hemoptysis": int(not _bool(p.get("hemoptysis"))),
        "no_estrogen": int(not _bool(p.get("estrogen_use"))),
        "no_surg/trauma_4w": int(not (_bool(p.get("recent_surgery_4w")) or _bool(p.get("recent_trauma_4w")))),
        "no_prior_vte": int(not _bool(p.get("prev_vte"))),
        "no_unilateral_swelling": int(not _bool(p.get("unilateral_leg_swelling"))),
    }
    return {"name":"PERC","passed": bool(all(crit.values()))}

def calc_pesi(p: Dict[str,Any]):
    age=_pick(p,["age"],int) or 0
    male = 10 if (str(p.get("sex") or "").upper().startswith("M")) else 0
    cancer = 30 if _bool(p.get("cancer_active")) else 0
    hf = 10 if _bool(p.get("heart_failure")) else 0
    lung = 10 if (_bool(p.get("copd")) or _bool(p.get("chronic_lung_disease"))) else 0
    hr = 20 if ((_pick(p,["hr","pulse"],float) or 0) >=110) else 0
    sbp = 30 if ((_pick(p,["sbp"],float) or 200) < 100) else 0
    rr = 20 if ((_pick(p,["rr"],float) or 0) >=30) else 0
    temp = 20 if ((_pick(p,["temp"],float) or 37) < 36) else 0
    altered = 60 if ((_pick(p,["gcs"],float) or 15) < 15) else 0
    sat = 20 if ((_pick(p,["sao2","spo2"],float) or 100) < 90) else 0
    score = age+male+cancer+hf+lung+hr+sbp+rr+temp+altered+sat
    klass = "I" if score<=65 else "II" if score<=85 else "III" if score<=105 else "IV" if score<=125 else "V"
    return {"name":"PESI","score": int(score), "class": klass}

def calc_spesi(p: Dict[str,Any]):
    comps = {
        "age>80": int((_pick(p,["age"],int) or 0) > 80),
        "cancer": int(_bool(p.get("cancer_active"))),
        "cardiopulm": int(_bool(p.get("heart_failure")) or _bool(p.get("copd")) or _bool(p.get("chronic_lung_disease"))),
        "hr>=110": int((_pick(p,["hr","pulse"],float) or 0) >= 110),
        "sbp<100": int((_pick(p,["sbp"],float) or 200) < 100),
        "o2<90": int((_pick(p,["sao2","spo2"],float) or 100) < 90),
    }
    return {"name":"sPESI","score": int(sum(comps.values()))}

def calc_sirs(p: Dict[str,Any]):
    crit = {
        "temp>38/<36": int(((_pick(p,["temp"],float) or 37)>38) or ((_pick(p,["temp"],float) or 37)<36)),
        "hr>90": int((_pick(p,["hr","pulse"],float) or 0) > 90),
        "rr>20/paco2<32": int(((_pick(p,["rr"],float) or 0) > 20) or ((_pick(p,["paco2"],float) or 100) < 32)),
        "wbc>12/<4/bands>10%": int(((_pick(p,["wbc"],float) or 7) > 12) or ((_pick(p,["wbc"],float) or 7) < 4) or ((_pick(p,["bands_pct"],float) or 0) > 10)),
    }
    return {"name":"SIRS","score": int(sum(crit.values()))}

def sepsis3_screen(v: Dict[str,Any], labs: Dict[str,Any], ctx: Dict[str,Any], sofa_min: Dict[str,Any], qsofa: Dict[str,Any]):
    sus = _bool(ctx.get("suspected_infection")); on_pressors=_bool(ctx.get("vasopressors"))
    mapv=_pick(v,["map"]); lact=_pick(labs,["lactate"])
    septic_shock = bool((mapv is not None and mapv<65) and (lact is not None and lact>2) and on_pressors)
    return {"name":"SEPSIS3","sepsis_flag": bool(sus and sofa_min["score"]>=2), "septic_shock": septic_shock}

def calc_marburg(p: Dict[str,Any]):
    sex=(str(p.get("sex") or "")[:1]).upper(); age=_pick(p,["age"],int)
    vasc=_bool(p.get("vasc_disease")); exert=_bool(p.get("exertional")); pt_thinks=_bool(p.get("patient_assumes_cardiac"))
    palp = p.get("palpation_reproducible"); not_repro = (palp is False)
    age_sex = ((sex=="M" and age is not None and age>=55) or (sex=="F" and age is not None and age>=65))
    sc = int(bool(age_sex)) + int(vasc) + int(exert) + int(pt_thinks) + int(bool(not_repro))
    return {"name":"MARBURG","score": int(sc)}

def calc_gbs(p: Dict[str,Any]):
    score=0
    urea=_pick(p,["urea_mmol_l","urea"]); bun=_pick(p,["bun_mg_dl"])
    if urea is None and bun is not None: urea=float(bun)/2.8
    hb_gL=_pick(p,["hb_g_l"]); 
    if hb_gL is None:
        hb_gdl=_pick(p,["hb","hb_g_dl"]); 
        if hb_gdl is not None: hb_gL=hb_gdl*10.0
    sbp=_pick(p,["sbp"]); hr=_pick(p,["hr","pulse"]); male = str(p.get("sex") or "").upper().startswith("M")
    if urea is not None:
        score += 2 if 6.5<=urea<=7.9 else 0; score += 3 if 8.0<=urea<=9.9 else 0
        score += 4 if 10.0<=urea<=25.0 else 0; score += 6 if urea>25.0 else 0
    if hb_gL is not None:
        if male:   score += (1 if 120<=hb_gL<=129 else 0) + (3 if 100<=hb_gL<=119 else 0) + (6 if hb_gL<100 else 0)
        else:      score += (1 if 100<=hb_gL<=119 else 0) + (6 if hb_gL<100 else 0)
    if sbp is not None: score += (1 if 100<=sbp<=109 else 0) + (2 if 90<=sbp<=99 else 0) + (3 if sbp<90 else 0)
    score += 1 if (hr is not None and hr>=100) else 0
    score += 1 if _bool(p.get("melena")) else 0
    score += 2 if _bool(p.get("syncope")) else 0
    score += 2 if _bool(p.get("hepatic_disease")) else 0
    score += 2 if _bool(p.get("cardiac_failure")) else 0
    return {"name":"GBS","score": int(score)}

def calc_child_pugh(p: Dict[str,Any]):
    bili=_pick(p,["bilirubin"]); alb=_pick(p,["albumin"]); inr=_pick(p,["inr"])
    asc=(p.get("ascites") or "").lower(); ence=(p.get("encephalopathy") or "").lower()
    sc=0; filled=0
    if bili is not None: sc+=(1 if bili<2 else 2 if bili<=3 else 3); filled+=1
    if alb  is not None: sc+=(1 if alb>3.5 else 2 if alb>=2.8 else 3); filled+=1
    if inr  is not None: sc+=(1 if inr<1.7 else 2 if inr<=2.3 else 3); filled+=1
    if asc:              sc+=(1 if asc.startswith("n") else 2 if asc.startswith(("mild","slight")) else 3); filled+=1
    if ence:             sc+=(1 if ence in {"none","0"} else 2 if any(x in ence for x in ["1","2","i","ii"]) else 3); filled+=1
    if filled<5: return {"name":"CHILD_PUGH","score_partial": int(sc), "class":"incomplete"}
    klass = "A" if sc<=6 else ("B" if sc<=9 else "C")
    return {"name":"CHILD_PUGH","score": int(sc), "class": klass}

def corrected_calcium(total_ca, albumin, units="mg/dL", normal_alb=None):
    if total_ca is None or albumin is None: return None
    if "mmol" in (units or "").lower():
        alb_gl = albumin if albumin>10 else albumin*10.0
        normal = 40.0 if normal_alb is None else float(normal_alb)
        return float(total_ca) + 0.02*(normal - alb_gl)
    normal = 4.0 if normal_alb is None else float(normal_alb)
    return float(total_ca) + 0.8*(normal - float(albumin))

def anion_gap(na, cl, hco3, k=None, albumin_gdl=None):
    if na is None or cl is None or hco3 is None: return None
    ag = (float(na)+(float(k) if k is not None else 0.0)) - (float(cl)+float(hco3))
    agc = ag + (2.5*(4.0 - float(albumin_gdl))) if albumin_gdl is not None else None
    return {"ag": _safe_round(ag,1), "ag_albumin_corrected": _safe_round(agc,1) if agc is not None else None}

# ---- HL7 (Troponin/D-dimer + INR/Bili/Albumin) ----
def parse_hl7_labs(hl7_text: str) -> Dict[str, Any]:
    troponin, ddimer = [], []
    others = {"bilirubin": None, "inr": None, "albumin": None}
    lines = re.split(r'[\r\n]+', (hl7_text or "").strip())
    for ln in lines:
        if not ln.strip(): continue
        parts = ln.split("|")
        obx3 = parts[3] if len(parts)>3 else ""
        obx5 = parts[5] if len(parts)>5 else ""
        obx6 = parts[6] if len(parts)>6 else ""
        obx14= parts[14] if len(parts)>14 else ""
        name=(obx3 or ln); val=obx5; unit=obx6
        ts=None
        mdt = re.search(r'(\d{8})(\d{6})?', obx14)
        if mdt:
            try: ts = datetime.strptime(mdt.group(1)+(mdt.group(2) or "000000"), "%Y%m%d%H%M%S")
            except Exception: ts=None
        try: v=float(str(val).replace(",", "."))
        except Exception:
            mnum=re.search(r'[-+]?\d*\.?\d+', str(val).replace(",", "."))
            v=float(mnum.group(0)) if mnum else None
        if not unit:
            munit=re.search(r'\[(.*?)\]', name)
            if munit: unit=munit.group(1)
        n=name.upper()
        if ("TROP" in n or "TROPONIN" in n) and v is not None: troponin.append((ts,v,unit or "ng/L"))
        if ("D-DIMER" in n or "DDIMER" in n or "D DIMER" in n) and v is not None: ddimer.append((ts,v,unit or "μg/L FEU"))
        if v is not None:
            if ("BILIRUBIN" in n or "BILI" in n) and others["bilirubin"] is None:
                others["bilirubin"] = (float(v)/17.1) if (unit or "").lower() in {"umol/l","µmol/l"} else float(v)
            if "INR" in n and others["inr"] is None: others["inr"]=float(v)
            if "ALBUMIN" in n and others["albumin"] is None:
                a=float(v)
                others["albumin"] = a/10.0 if (unit or "").lower() in {"g/l","g l","gl"} else a
    return {"troponin": troponin, "d_dimer": ddimer, "others": others}

# ---- Bundle / one-liners ----
def phase2_bundle(vitals: Dict[str,Any], labs: Dict[str,Any], context: Optional[Dict[str,Any]]=None) -> Dict[str,Any]:
    context=context or {}
    age=_pick({**vitals, **context},["age"],int)
    pregnant=bool(context.get("pregnant", False))

    s_qsofa=calc_qsofa(vitals)
    s_mews =calc_mews(vitals)
    s_heart=calc_heart({**vitals, **labs})
    s_grace=calc_grace_coarse({**vitals, **labs})
    s_sofa =calc_sofa_min(vitals, labs)

    d_val=_pick(labs,["d_dimer","ddimer","d-dimer","d_dimer_feu_ug_l"]); d_unit=labs.get("d_dimer_unit") or "μg/L FEU"
    d_assess = assess_d_dimer(d_val, d_unit, age, pregnant) if d_val is not None else {"available": False}

    series=[]
    for it in labs.get("troponin_series") or []:
        if isinstance(it, dict):
            ts=None
            if it.get("time"):
                try: ts=datetime.fromisoformat(str(it["time"]).replace("Z",""))
                except Exception: ts=None
            series.append((ts, _pick(it,["value","val"]), it.get("unit","ng/L")))
    t_assess = assess_troponin_delta(series) if series else {"available": False}

    wells_pe  = calc_wells_pe({**vitals, **labs, **context})
    wells_dvt = calc_wells_dvt({**vitals, **labs, **context})
    perc      = calc_perc({**vitals, **labs, **context})
    pesi      = calc_pesi({**vitals, **labs, **context})
    spesi     = calc_spesi({**vitals, **labs, **context})
    sirs      = calc_sirs({**vitals, **labs})
    sepsis3   = sepsis3_screen(vitals, labs, context, s_sofa, s_qsofa)
    marburg   = calc_marburg({**vitals, **labs, **context})
    gbs       = calc_gbs({**vitals, **labs, **context})
    childpugh = calc_child_pugh({**vitals, **labs, **context})
    ca_corr   = corrected_calcium(_pick(labs,["calcium","ca","calcium_mg_dl"]), _pick(labs,["albumin","alb","albumin_g_dl"]), units="mg/dL")
    ag_val    = anion_gap(_pick(labs,["na","sodium"]), _pick(labs,["cl","chloride"]), _pick(labs,["hco3","bicarbonate"]),
                          k=_pick(labs,["k","potassium"]), albumin_gdl=_pick(labs,["albumin","alb","albumin_g_dl"]))

    ones=[]
    ones.append(f"qSOFA={s_qsofa['score']}  MEWS={s_mews['score']}  GRACE(coarse)={s_grace['score']}")
    ones.append(f"HEART={s_heart['score']}  SOFA(min)={s_sofa['score']}")
    if d_assess.get("available"):
        ones.append(f"D-dimer {int(d_assess['value_ug_per_l'])} vs thr {int(d_assess['thr_ug_per_l'])} μg/L → {'OK' if d_assess['ok_below_thr'] else 'High'}")
    if t_assess.get("available"):
        da=_safe_round(t_assess['delta_abs'],1); dp=_safe_round(t_assess['delta_pct'],1) if t_assess['delta_pct'] is not None else None
        ones.append(f"Troponin Δ {da} ng/L ({dp}%) → {'FLAG' if t_assess['flag'] else 'ok'}")
    ones.append(f"Wells-PE={_safe_round(wells_pe['score'],1)} ({wells_pe['tier2']})  Wells-DVT={wells_dvt['score']} ({wells_dvt['tier2']})")
    ones.append(f"PERC={'pass' if perc['passed'] else 'fail'}  sPESI={spesi['score']}  PESI={pesi['class']}/{pesi['score']}")
    ones.append(f"SIRS={sirs['score']}  Sepsis3: {'YES' if sepsis3['sepsis_flag'] else 'no'}  Shock: {'YES' if sepsis3['septic_shock'] else 'no'}")
    ones.append(f"MARBURG={marburg['score']}/5  GBS={gbs['score']}")
    if childpugh.get("class") == "incomplete":
        ones.append("Child-Pugh incomplete (need 5/5 inputs)")
    else:
        ones.append(f"Child-Pugh {childpugh['class']} ({childpugh['score']})")
    if ca_corr is not None: ones.append(f"Corrected Ca={_safe_round(ca_corr,2)} mg/dL")
    if ag_val is not None:
        ab = f", AGcorr={ag_val['ag_albumin_corrected']}" if ag_val.get("ag_albumin_corrected") is not None else ""
        ones.append(f"Anion gap={ag_val['ag']}{ab}")

    out = {
        "one_liners": ones,
        "scores": {
            "qsofa": s_qsofa["score"], "mews": s_mews["score"], "heart": s_heart["score"], "grace_coarse": s_grace["score"],
            "sofa_min": s_sofa["score"], "wells_pe": wells_pe["score"], "wells_dvt": wells_dvt["score"],
            "pesi": pesi["score"], "spesi": spesi["score"], "sirs": sirs["score"], 
            "sepsis3": int(sepsis3["sepsis_flag"]), "gbs": gbs["score"], "marburg": marburg["score"],
            "child_pugh": childpugh.get("score") or childpugh.get("score_partial")
        },
        "labs": {
            "d_dimer": d_val, "d_dimer_thr": d_assess.get("thr_ug_per_l"),
            "trop_prev": t_assess.get("prev"), "trop_curr": t_assess.get("curr"),
            "trop_delta": t_assess.get("delta_abs"), "trop_delta_pct": t_assess.get("delta_pct"), "trop_flag": t_assess.get("flag")
        }
    }
    return out

# ---- UI (ipywidgets) + HL7 merge ----
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W, pandas as pd
        # text areas
        vitals_in = W.Textarea(
            value='{"rr":24,"sbp":95,"gcs":14,"hr":120,"temp":38.6,"avpu":"V","map":65,"age":68,"sex":"M","sao2":97}',
            description="Vitals JSON", layout=W.Layout(width="100%", height="90px")
        )
        labs_in = W.Textarea(
            value='{"d_dimer":780,"d_dimer_unit":"μg/L FEU","troponin_series":[{"time":"2025-08-20T05:00:00Z","value":18,"unit":"ng/L"},{"time":"2025-08-20T09:00:00Z","value":78,"unit":"ng/L"}],"creatinine":1.8,"platelets":95,"albumin":2.8,"calcium":7.9,"na":138,"cl":105,"hco3":20,"bilirubin":2.1,"inr":1.9}',
            description="Labs JSON", layout=W.Layout(width="100%", height="130px")
        )
        ctx_in = W.Textarea(
            value='{"pregnant": false, "suspected_infection": true, "vasopressors": false, "pe_most_likely": true, "dvt_signs": false, "ascites":"mild","encephalopathy":"1-2"}',
            description="Context JSON", layout=W.Layout(width="100%", height="90px")
        )

        # HEART 0–2
        heart_hist = W.Dropdown(options=[0,1,2], value=0, description="HEART: History")
        heart_ecg  = W.Dropdown(options=[0,1,2], value=0, description="HEART: ECG")
        heart_risk = W.Dropdown(options=[0,1,2], value=0, description="HEART: Risk")

        # PE / PERC flags
        chk_pe_most = W.Checkbox(description="PE most likely", value=True)
        chk_dvt_signs = W.Checkbox(description="DVT signs", value=False)
        chk_immob = W.Checkbox(description="Immobilized / recent surgery (4w)", value=False)
        chk_prev_vte = W.Checkbox(description="Prior VTE", value=False)
        chk_hemo = W.Checkbox(description="Hemoptysis", value=False)
        chk_cancer = W.Checkbox(description="Active malignancy", value=False)
        chk_estrogen = W.Checkbox(description="Estrogen use (PERC)", value=False)
        chk_trauma = W.Checkbox(description="Recent trauma (4w, PERC)", value=False)
        chk_unilat = W.Checkbox(description="Unilateral leg swelling (PERC)", value=False)

        # Wells-DVT flags
        dvt_cancer     = W.Checkbox(description="Active cancer", value=False)
        dvt_paresis    = W.Checkbox(description="Paresis / plaster cast", value=False)
        dvt_bed_surg   = W.Checkbox(description="Bedridden ≥3d / surgery ≤12w", value=False)
        dvt_tender     = W.Checkbox(description="Deep vein tenderness", value=False)
        dvt_entire     = W.Checkbox(description="Entire leg swollen", value=False)
        dvt_calf3      = W.Checkbox(description="Calf swelling >3 cm", value=False)
        dvt_edema      = W.Checkbox(description="Pitting edema (symptomatic leg)", value=False)
        dvt_collat     = W.Checkbox(description="Collateral non-varicose", value=False)
        dvt_prev       = W.Checkbox(description="Previous DVT", value=False)
        dvt_alt_dx     = W.Checkbox(description="Alternative dx as likely (subtract)", value=False)

        # HL7 paste
        hl7_in = W.Textarea(value='''OBX|1|NM|BILIRUBIN TOTAL||36|umol/L|||N||F|||20250820090000
OBX|2|NM|INR||1.9|||N||F|||20250820090000
OBX|3|NM|ALBUMIN||28|g/L|||N||F|||20250820090000
OBX|4|NM|D-DIMER||780|ug/L|||H||F|||20250820090000
OBX|5|NM|TROPONIN T||78|ng/L|||H||F|||20250820090000
OBX|6|NM|TROPONIN T||18|ng/L|||N||F|||20250820050000''',
                            description="HL7 paste", layout=W.Layout(width="100%", height="140px"))

        btn_apply_hl7 = W.Button(description="Apply HL7 → Labs JSON")
        btn_compute   = W.Button(description="Compute + Log")
        out = W.Output()

        def on_apply(_):
            with out:
                try:
                    labs = json.loads(labs_in.value or "{}")
                except Exception as e:
                    out.clear_output(); print("[labs parse error]", e); return
                parsed = parse_hl7_labs(hl7_in.value or "")
                oth = parsed.get("others") or {}
                for k in ["bilirubin","inr","albumin"]:
                    if oth.get(k) is not None:
                        labs[k] = oth[k]
                labs_in.value = json.dumps(labs, ensure_ascii=False)
                out.clear_output(); print("HL7 fields merged into Labs JSON:", {k:labs.get(k) for k in ["bilirubin","inr","albumin"]})

        def on_compute(_):
            with out:
                out.clear_output()
                try:
                    vitals = json.loads(vitals_in.value or "{}")
                    labs   = json.loads(labs_in.value or "{}")
                    ctx    = json.loads(ctx_in.value or "{}")
                except Exception as e:
                    print("[parse error]", e); return

                # HEART components
                vitals["heart_history"] = int(heart_hist.value)
                vitals["heart_ecg"]     = int(heart_ecg.value)
                vitals["heart_risk"]    = int(heart_risk.value)

                # Map PE/PERC + DVT flags
                ctx.update({
                    "pe_most_likely": chk_pe_most.value,
                    "dvt_signs": chk_dvt_signs.value,
                    "immobilized": chk_immob.value,
                    "recent_surgery_4w": chk_immob.value,
                    "prev_vte": chk_prev_vte.value,
                    "hemoptysis": chk_hemo.value,
                    "cancer_active": chk_cancer.value or ctx.get("cancer_active", False),
                    "estrogen_use": chk_estrogen.value,
                    "recent_trauma_4w": chk_trauma.value,
                    "unilateral_leg_swelling": chk_unilat.value,
                    # Wells-DVT details
                    "paresis": dvt_paresis.value,
                    "plaster_cast": dvt_paresis.value,
                    "bedridden_3d": dvt_bed_surg.value,
                    "surgery_12w": dvt_bed_surg.value,
                    "deep_vein_tenderness": dvt_tender.value,
                    "entire_leg_swollen": dvt_entire.value,
                    "calf_swelling_gt3cm": dvt_calf3.value,
                    "pitting_edema": dvt_edema.value,
                    "collateral_nonvaricose": dvt_collat.value,
                    "prev_dvt": dvt_prev.value,
                    "alt_dx_as_likely": dvt_alt_dx.value,
                })

                bundle = phase2_bundle(vitals, labs, ctx)
                _append_event({"type":"phase2_bundle","bundle": bundle})
                print("\n".join(bundle["one_liners"]))
                print("\nLogged to:", EVENT_LOG_PATH)

        btn_apply_hl7.on_click(on_apply)
        btn_compute.on_click(on_compute)

        display(W.VBox([
            W.HTML("<b>Phase-2 calculators — optimized UI (PE/DVT/PERC, PESI/sPESI, SIRS/Sepsis3, HEART, GBS, Child-Pugh)</b>"),
            W.HBox([vitals_in, labs_in]),
            W.HBox([heart_hist, heart_ecg, heart_risk]),
            W.HTML("<b>PE / PERC flags</b>"),
            W.HBox([chk_pe_most, chk_dvt_signs, chk_immob, chk_prev_vte, chk_hemo, chk_cancer]),
            W.HBox([chk_estrogen, chk_trauma, chk_unilat]),
            W.HTML("<b>Wells-DVT flags</b>"),
            W.HBox([dvt_cancer, dvt_paresis, dvt_bed_surg, dvt_tender, dvt_entire]),
            W.HBox([dvt_calf3, dvt_edema, dvt_collat, dvt_prev, dvt_alt_dx]),
            ctx_in,
            W.HBox([btn_compute, btn_apply_hl7]),
            W.HTML("<b>HL7 Labs (Trop / D-Dimer / INR / Bili / Albumin)</b>"),
            hl7_in,
            out
        ]))
    except Exception as e:
        print("Phase-2 UI unavailable:", e)
else:
    # Text-mode smoke test (runs if RUN_UI=False)
    vit = {"rr":24,"sbp":95,"gcs":14,"hr":120,"temp":38.6,"avpu":"V","map":65,"age":68,"sex":"M","sao2":97,
           "heart_history":0,"heart_ecg":0,"heart_risk":0}
    lab = {"d_dimer":780,"d_dimer_unit":"μg/L FEU",
           "troponin_series":[{"time":"2025-08-20T05:00:00Z","value":18,"unit":"ng/L"},{"time":"2025-08-20T09:00:00Z","value":78,"unit":"ng/L"}],
           "creatinine":1.8,"platelets":95,"albumin":2.8,"calcium":7.9,"na":138,"cl":105,"hco3":20,"bilirubin":2.1,"inr":1.9}
    ctx = {"pregnant": True, "suspected_infection": True, "vasopressors": False, "pe_most_likely": True, "dvt_signs": False,
           "ascites":"mild","encephalopathy":"1-2"}
    b = phase2_bundle(vit, lab, ctx)
    _append_event({"type":"phase2_bundle_smoke","bundle": b})
    print("\n".join(b["one_liners"]))
    print("Logged to:", EVENT_LOG_PATH)


In [ ]:
# Force-sync all "Context JSON" textareas and the reference used by YEARS
import json, ipywidgets as W, gc

def _update_all_context_widgets(flag=True):
    new_val = None
    # If a ctx_in exists in this scope, start from it
    try:
        d = json.loads(ctx_in.value or "{}")
    except Exception:
        d = {}
    d["pregnant"] = bool(flag)
    new_val = json.dumps(d, ensure_ascii=False)

    # Update any Textarea whose description starts with "Context JSON"
    n = 0
    for obj in gc.get_objects():
        try:
            if isinstance(obj, W.Textarea) and (obj.description or "").startswith("Context JSON"):
                obj.value = new_val
                n += 1
        except Exception:
            pass
    print(f"Updated {n} Context JSON widget(s) →", new_val)

    # Make sure the YEARS cell reads the same widget reference
    global ctx_src  # used inside the YEARS cell handler
    try:
        ctx_src = ctx_in
        print("ctx_src → ctx_in (bound)")
    except NameError:
        print("ctx_in not in scope; YEARS will still read its own ctx_src if present.")

_update_all_context_widgets(flag=True)


In [ ]:

# === Pregnancy-adapted YEARS pathway (drop-in) ===
# Uses the Phase-2 panel's Context/Labs if available; otherwise exposes its own minimal text areas.
# Logs to CONFIG['EVENT_LOG_PATH'] as {"type": "pregnancy_years", ...}

from datetime import datetime, timezone
import json

def assess_pregnancy_years(ctx, labs):
    # Preg-adapted YEARS: three items — DVT signs, hemoptysis, 'PE most likely'.
    # If none present → D-dimer threshold 1000 μg/L FEU; else threshold 500 μg/L FEU.
    preg = bool(ctx.get("pregnant", False))
    items = {
        "dvt_signs": bool(ctx.get("dvt_signs", False)),
        "hemoptysis": bool(ctx.get("hemoptysis", False)),
        "pe_most_likely": bool(ctx.get("pe_most_likely", False)),
    }
    count = sum(int(v) for v in items.values())
    d_val = None
    for k in ("d_dimer","ddimer","d-dimer","d_dimer_feu_ug_l"):
        if k in labs and labs[k] not in (None, ""):
            try:
                d_val = float(str(labs[k]).replace(",", "."))
                break
            except Exception:
                pass
    d_unit = (labs.get("d_dimer_unit") or "μg/L FEU").lower()
    if d_val is None:
        return {"applies": preg, "needs_imaging": True, "reason": "missing D-dimer", "items": items, "items_count": count}
    d_ug = float(d_val)*1000.0 if "mg/l" in d_unit else float(d_val)
    thr = 1000.0 if count==0 else 500.0
    ruleout = bool(d_ug < thr)
    return {
        "applies": preg, "items": items, "items_count": count,
        "d_dimer_ug_l": d_ug, "threshold_ug_l": thr, "rule_out": ruleout,
        "note": "If DVT signs present, compression ultrasound is recommended before applying YEARS."
    }

# UI that reuses Phase-2 widget inputs if present (vitals_in / labs_in / ctx_in).
try:
    _probe = vitals_in  # noqa: F401
    _has_phase2_ui = True
except NameError:
    _has_phase2_ui = False

if CONFIG.get("RUN_UI", False):
    try:
        import ipywidgets as W
        if _has_phase2_ui:
            labs_src = labs_in
            ctx_src  = ctx_in
            src_note_ctx = W.HTML('<i>Using Context JSON from Phase-2 panel</i>')
            src_note_labs = W.HTML('<i>Using Labs JSON from Phase-2 panel</i>')
        else:
            labs_src = W.Textarea(value='{"d_dimer":780,"d_dimer_unit":"μg/L FEU"}', description="Labs JSON", layout=W.Layout(width="100%", height="80px"))
            ctx_src  = W.Textarea(value='{"pregnant": true, "pe_most_likely": true, "dvt_signs": false, "hemoptysis": false}', description="Context JSON", layout=W.Layout(width="100%", height="80px"))
            src_note_ctx = src_note_labs = W.HTML('')
        out = W.Output()
        btn = W.Button(description="Compute Pregnancy YEARS + Log")
        warn = W.HTML("<small><b>Note:</b> PERC is not validated in pregnancy; use the YEARS pathway below.</small>")

        def _on_click(_):
            with out:
                out.clear_output()
                try:
                    labs = json.loads(labs_src.value if hasattr(labs_src, 'value') else labs_src)
                    ctx  = json.loads(ctx_src.value if hasattr(ctx_src, 'value') else ctx_src)
                except Exception as e:
                    print("[parse error]", e); return
                res = assess_pregnancy_years(ctx, labs)
                if not res.get("applies", False):
                    print("Pregnancy YEARS: not applicable (pregnant flag is false).")
                else:
                    msg = f"Pregnancy YEARS: items={res['items_count']} → D-dimer thr {int(res['threshold_ug_l'])} μg/L; value {int(res['d_dimer_ug_l'])} → "
                    msg += ("RULE-OUT" if res["rule_out"] else "IMAGING")
                    print(msg)
                try:
                    _append_event({"type":"pregnancy_years", "result": res})
                    print("Logged to:", CONFIG.get("EVENT_LOG_PATH"))
                except Exception as e:
                    print("[log error]", e)

        btn.on_click(_on_click)

        display(W.VBox([
            W.HTML("<b>Pregnancy-adapted YEARS pathway</b>"),
            warn,
            src_note_ctx, src_note_labs,
            btn, out
        ]))
    except Exception as e:
        print("YEARS UI unavailable:", e)

# Text-mode smoke if RUN_UI is off
if not CONFIG.get("RUN_UI", False):
    labs = {"d_dimer": 780, "d_dimer_unit": "μg/L FEU"}
    ctx = {"pregnant": True, "pe_most_likely": True, "dvt_signs": False, "hemoptysis": False}
    res = assess_pregnancy_years(ctx, labs)
    msg = f"Pregnancy YEARS (text-mode): items={res.get('items_count',0)}; thr={int(res.get('threshold_ug_l',0))} μg/L; value={int(res.get('d_dimer_ug_l',0))} → "
    msg += ("RULE-OUT" if res.get("rule_out") else "IMAGING")
    print(msg)
    try:
        _append_event({"type":"pregnancy_years_smoke", "result": res})
        print("Logged to:", CONFIG.get("EVENT_LOG_PATH"))
    except Exception as e:
        print("[log error]", e)


In [ ]:
!tail -n 5 {CONFIG['EVENT_LOG_PATH']}


In [ ]:

# UI: Medication Plan Import (scan or paste), Interaction/Allergy checks — inline, no sidecars
if CONFIG.get("RUN_UI") and CONFIG.get("RUN_MEDS"):
    try:
        import ipywidgets as W
        import pandas as pd
        from IPython.display import display

        patient = W.Text(value="", description="Patient ID")
        path_in = W.Text(value="", placeholder="/mnt/data/scan.png or .pdf", description="Scan path")
        ocr_btn = W.Button(description="Run OCR from scan")
        paste = W.Textarea(value="", placeholder="Paste BMP text here…", description="Text", layout=W.Layout(width="90%", height="120px"))
        parse_btn = W.Button(description="Parse & Check")
        out = W.Output()

        def _to_df(items: list[dict], cols=None):
            if not items: return pd.DataFrame()
            df = pd.DataFrame(items)
            return df if cols is None else df.reindex(columns=cols, fill_value="")

        def on_ocr(_):
            with out:
                out.clear_output()
                res = meds_pipeline_from_scan(path_in.value.strip(), patient_id=patient.value.strip() or None)
                if "error" in res:
                    print("OCR error:", res["error"])
                    return
                mdf = _to_df(res["meds"], cols=["name","strength","unit","freq","route","prn","atc","raw"])
                idf = _to_df(res["interactions"])
                adf = _to_df(res["allergy_warnings"])
                print("Parsed meds:"); display(mdf)
                if not idf.empty: print("\nInteractions:"); display(idf)
                if not adf.empty: print("\nAllergy warnings:"); display(adf)

        def on_parse(_):
            with out:
                out.clear_output()
                res = meds_pipeline_from_text(paste.value, patient_id=patient.value.strip() or None)
                mdf = _to_df(res["meds"], cols=["name","strength","unit","freq","route","prn","atc","raw"])
                idf = _to_df(res["interactions"])
                adf = _to_df(res["allergy_warnings"])
                print("Parsed meds:"); display(mdf)
                if not idf.empty: print("\nInteractions:"); display(idf)
                if not adf.empty: print("\nAllergy warnings:"); display(adf)

        ocr_btn.on_click(on_ocr)
        parse_btn.on_click(on_parse)

        display(W.VBox([
            W.HTML("<b>Medication Plan Import (scan or paste)</b>"),
            patient,
            W.HBox([path_in, ocr_btn]),
            paste,
            parse_btn,
            out
        ]))
    except Exception as e:
        print("Meds UI unavailable:", e)
else:
    print("Meds UI deferred… set CONFIG['RUN_UI']=True and CONFIG['RUN_MEDS']=True.")


In [ ]:

# === BMP (Bundeseinheitlicher Medikationsplan) scanner + importer (inline, no sidecars) ===
# Purpose: decode DataMatrix barcode from the printed BMP, parse XML "Carriersegment", extract meds,
#          run allergy cross-reactivity + interaction heuristics, and stage import payloads.
# Guards: RUN_UI controls widget panel; no background jobs.
from __future__ import annotations
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, Any, List, Optional, Tuple
import json, io

# Soft deps
def _try_imports():
    dm = None
    try:
        from pylibdmtx.pylibdmtx import decode as dm_decode  # DataMatrix
        from PIL import Image
        dm = (dm_decode, Image)
    except Exception:
        dm = None
    return dm

def _decode_datamatrix_from_image(path: str) -> Optional[str]:
    dm = _try_imports()
    if dm is None:
        return None
    dm_decode, Image = dm
    try:
        img = Image.open(path)
        res = dm_decode(img)
        if res:
            # Choose the longest decoded payload
            payload = max(res, key=lambda r: len(r.data)).data
            try:
                return payload.decode("iso-8859-1", errors="ignore")
            except Exception:
                return payload.decode("utf-8", errors="ignore")
    except Exception:
        return None
    return None

# Minimal XML parser for the BMP "Carriersegment" (per KBV spec: XML, root <MP ...>)
def _parse_bmp_xml(xml_text: str) -> Dict[str, Any]:
    import xml.etree.ElementTree as ET
    meds = []
    meta = {}
    try:
        root = ET.fromstring(xml_text.strip())
    except Exception as e:
        return {"ok": False, "error": f"XML parse failed: {e}"}
    # Root attributes include version v="022" etc.
    meta["root_tag"] = root.tag
    meta["version"] = root.attrib.get("v")
    # Heuristic: medication entries appear as child elements (e.g., <M/> or <A/> etc. depending on schema);
    # We'll scan for elements carrying PZN ("P" or "PZN"), substance ("WST"), strength, dose instructions.
    for el in root.iter():
        tag = el.tag
        at = el.attrib or {}
        # Look for attributes commonly used in samples
        pzn = at.get("P") or at.get("PZN") or None
        name = at.get("HN") or at.get("NAME") or at.get("TXT") or None  # Handelsname / text
        wst  = at.get("WST") or at.get("WS") or None  # Wirkstoff(e)
        strg = at.get("ST") or at.get("STR") or at.get("STAERKE") or None
        dose = at.get("DA") or at.get("DOS") or at.get("DOSE") or None
        freq = at.get("FREQ") or at.get("FRQ") or None
        route = at.get("ROUTE") or at.get("ANW") or None
        form = at.get("DAR") or at.get("DARZ") or None  # Darreichungsform (IFA code)
        if any([pzn, name, wst, strg, dose]):
            meds.append({
                "pzn": pzn,
                "name": name,
                "substance": wst,
                "strength": strg,
                "dose": dose,
                "freq": freq,
                "route": route,
                "form": form,
                "_tag": tag,
                "_raw": at
            })
    # Deduplicate meds by pzn + name + substance
    seen = set()
    uniq = []
    for m in meds:
        key = (m.get("pzn"), m.get("name"), m.get("substance"), m.get("strength"), m.get("dose"))
        if key not in seen:
            seen.add(key); uniq.append(m)
    return {"ok": True, "meta": meta, "meds": uniq}

# Allergy + interaction heuristics (non-decisional, warnings only)
ALLERGY_GROUPS = {
    "penicillin": {"aliases": ["penicillin", "amoxicillin", "ampicillin", "oxacillin", "piperacillin", "flucloxacillin"]},
    "cephalosporin": {"aliases": ["cephalosporin", "cefaclor", "cefazolin", "cefalexin", "cefuroxim", "ceftriaxon", "ceftazidim", "cefpodoxim"]},
    "sulfonamide_abx": {"aliases": ["sulfamethoxazole", "trimethoprim-sulfamethoxazole", "co-trimoxazole"]},
    "nsaid": {"aliases": ["ibuprofen","diclofenac","naproxen","aspirin","ketorolac","indometacin","piroxicam"]},
    "opioid": {"aliases": ["morphin","oxycodon","fentanyl","hydromorphon","tramadol","codein","tilidin"]},
}

CROSS_REACTIVITY_RULES = [
    # Rule format: (allergy_group, suspect_group, note)
    ("penicillin", "cephalosporin", "Beta-Lactam cross-reactivity possible; clinical relevance depends on generation/side-chain; review carefully."),
    ("nsaid", "nsaid", "NSAID hypersensitivity often cross-reactive across nonselective NSAIDs; consider COX-2 selection if appropriate."),
    ("sulfonamide_abx", "sulfonamide_abx", "Avoid sulfonamide antibiotics if sulfonamide antibiotic allergy is present."),
]

# Interaction rules (coarse). Pattern match on substance or name substrings (casefold); non-decisional warnings.
INTERACTION_RULES = [
    (["warfarin","phenprocoumon","acenocoumarol"], ["nsaid","ibuprofen","diclofenac","naproxen","aspirin"],
     "Anticoagulant + NSAID → bleeding risk; consider gastroprotection/alternatives; monitor INR if VKA."),
    (["warfarin","phenprocoumon","acenocoumarol"], ["macrolid","erythromycin","clarithromycin","azithromycin","fluoroquinolon","ciprofloxacin","levofloxacin"],
     "VKA + certain antibiotics → ↑INR/bleeding; monitor closely."),
    (["simvastatin","lovastatin"], ["clarithromycin","erythromycin","itraconazol","ketoconazol","posaconazol","voriconazol"],
     "Statin (CYP3A4) + strong inhibitor → ↑rhabdomyolysis risk; consider hold/switch."),
    (["ssri","citalopram","escitalopram","sertralin","fluoxetin","paroxetin","duloxetin","venlafaxin"], ["maoi","triptan","linezolid"],
     "Serotonergic combo → serotonin syndrome risk; avoid/monitor."),
    (["ace","ramipril","enalapril","lisinopril","sartan","valsartan","losartan"], ["spironolacton","eplerenon","amilorid","triamteren","kalium"],
     "RAAS blocker + K-sparing / potassium → hyperkalaemia risk; check K+/renal function."),
    (["doac","apixaban","rivaroxaban","edoxaban","dabigatran"], ["dual antiplatelet","clopidogrel","prasugrel","ticagrelor","aspirin"],
     "Anticoagulant + antiplatelet(s) → bleeding risk; verify indication and duration."),
    (["qt","amiodaron","sotalol","haloperidol","ciprofloxacin","levofloxacin","clarithromycin","erythromycin","citalopram","escitalopram"], 
     ["qt","amiodaron","sotalol","haloperidol","ciprofloxacin","levofloxacin","clarithromycin","erythromycin","citalopram","escitalopram"],
     "Potential additive QT prolongation; assess ECG/QT and risk factors.")
]

def _norm(s: Optional[str]) -> str:
    return (s or "").strip().casefold()

def _hit(item_text: str, keys: List[str]) -> bool:
    t = item_text
    for k in keys:
        if k in t:
            return True
    return False

def _med_text(m: Dict[str, Any]) -> str:
    parts = [m.get("name") or m.get("substance") or "", m.get("strength") or "", m.get("dose") or ""]
    return " ".join([p for p in parts if p]).strip()

def check_allergy_cross_reactivity(allergies: List[str], meds: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    warnings = []
    alg_norm = [_norm(a) for a in allergies]
    # Map allergies to groups
    has_group = {g: any(any(alias in a for alias in v["aliases"]) for a in alg_norm) for g,v in ALLERGY_GROUPS.items()}
    for m in meds:
        text = _norm(_med_text(m))
        for (ag, sg, note) in CROSS_REACTIVITY_RULES:
            if has_group.get(ag, False):
                # Does med fall in suspect group?
                suspects = ALLERGY_GROUPS.get(sg, {}).get("aliases", [])
                if _hit(text, [s.casefold() for s in suspects]):
                    warnings.append({"type":"allergy_crossreactivity", "against": ag, "med": m, "note": note})
    return warnings

def check_interactions(meds: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    # Create flat list of med strings to match
    texts = [_norm(_med_text(m)) for m in meds]
    pair_warnings = []
    n = len(meds)
    for i in range(n):
        for j in range(i+1, n):
            t1, t2 = texts[i], texts[j]
            for A, B, note in INTERACTION_RULES:
                if (_hit(t1, [a.casefold() for a in A]) and _hit(t2, [b.casefold() for b in B])) or \
                   (_hit(t2, [a.casefold() for a in A]) and _hit(t1, [b.casefold() for b in B])):
                    pair_warnings.append({"type":"interaction", "med1": meds[i], "med2": meds[j], "note": note})
    # Deduplicate warnings by note + meds
    uniq = []
    seen = set()
    for w in pair_warnings:
        key = (w["note"], _med_text(w["med1"]), _med_text(w["med2"]))
        if key not in seen:
            uniq.append(w); seen.add(key)
    return uniq

# UI
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W, pandas as pd
        from IPython.display import display, HTML

        img_path = W.Text(description="Plan image", placeholder="/path/to/photo_or_scan.png")
        paste = W.Textarea(description="Paste payload", placeholder="(Optional) Paste decoded XML or BK/UKF text here…", layout=W.Layout(width="100%", height="120px"))
        allergies_in = W.Text(description="Allergies", placeholder="e.g. penicillin; ibuprofen", layout=W.Layout(width="60%"))
        run = W.Button(description="Scan/Parse", button_style="primary")
        out = W.Output()

        def on_run(_):
            with out:
                out.clear_output()
                payload = None
                if img_path.value:
                    payload = _decode_datamatrix_from_image(img_path.value)
                    if payload:
                        print("Decoded DataMatrix payload (first 200 chars):", payload[:200], "…")
                if not payload and paste.value.strip():
                    payload = paste.value.strip()
                    print("Using pasted payload.")
                if not payload:
                    print("No payload found. Install pylibdmtx or paste XML from the 2D code.")
                    return
                # If payload looks like XML (<MP ...>), parse; else just show raw
                if "<" in payload and ">" in payload:
                    res = _parse_bmp_xml(payload)
                    if not res.get("ok"):
                        print("Parse error:", res.get("error"))
                        return
                    meds = res["meds"]
                    if not meds:
                        print("No medication entries detected in XML. Showing raw payload snippet:\n", payload[:400], "…")
                        return
                    df = pd.DataFrame([{
                        "PZN": m.get("pzn"),
                        "Name/Substance": m.get("name") or m.get("substance"),
                        "Strength": m.get("strength"),
                        "Dose": m.get("dose"),
                        "Route": m.get("route"),
                        "Form": m.get("form"),
                    } for m in meds])
                    display(HTML("<b>Parsed medications (from BMP)</b>"))
                    display(df.style.hide(axis='index'))

                    allergies = [a.strip() for a in allergies_in.value.split(";") if a.strip()]
                    alg_warn = check_allergy_cross_reactivity(allergies, meds) if allergies else []
                    int_warn = check_interactions(meds)

                    if alg_warn or int_warn:
                        print("\nWarnings (review; not therapeutic decisions):")
                        for w in alg_warn:
                            print(f" - Allergy cross-reactivity ({w['against']}): { _med_text(w['med']) } → {w['note']}")
                        for w in int_warn:
                            print(f" - Interaction: { _med_text(w['med1']) } + { _med_text(w['med2']) } → {w['note']}")
                    else:
                        print("No heuristic warnings found (rule-base is limited; always review clinically).")

                    # Stage import payload (FHIR-like MedicationStatement draft)
                    bundle = {
                        "resourceType": "Bundle",
                        "type": "collection",
                        "timestamp": datetime.now(timezone.utc).isoformat(),
                        "entry": []
                    }
                    for m in meds:
                        entry = {
                            "resource": {
                                "resourceType": "MedicationStatement",
                                "status": "active",
                                "medicationCodeableConcept": {
                                    "text": m.get("name") or m.get("substance") or (m.get("pzn") and f"PZN {m['pzn']}") or "Unknown"
                                },
                                "dosage": [{
                                    "text": " ".join([x for x in [m.get("dose"), m.get("freq"), m.get("route")] if x])
                                }]
                            }
                        }
                        # Attach PZN as identifier if we have it
                        if m.get("pzn"):
                            entry["resource"]["medicationCodeableConcept"]["coding"] = [{
                                "system": "https://fhir.kbv.de/CodeSystem/KBV_CS_VS_PZN",
                                "code": m["pzn"]
                            }]
                        bundle["entry"].append(entry)
                    # Write draft bundle to /mnt/data for later Orbis integration
                    out_path = Path("/mnt/data/bmp_import_bundle.json")
                    out_path.write_text(json.dumps(bundle, ensure_ascii=False, indent=2))
                    print("\nDraft import bundle written to:", str(out_path))

                else:
                    print("Payload does not look like XML. Showing first 400 chars:\n", payload[:400], "…")
        run.on_click(on_run)

        display(W.VBox([
            W.HTML("<b>Bundeseinheitlicher Medikationsplan (BMP) → decode → parse → warn → stage import</b>"),
            img_path, paste, allergies_in, run, out
        ]))

    except Exception as e:
        print("BMP scanner UI unavailable:", e)
else:
    print("BMP scanner deferred… set CONFIG['RUN_UI']=True.")


In [ ]:

# === Medication Plan Import + Checks (inline demo) ===
from __future__ import annotations
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime, timezone
import re, json

EVENT_LOG = Path(CONFIG["EVENT_LOG_PATH"])

def _append_event(ev: Dict[str, Any]):
    EVENT_LOG.parent.mkdir(parents=True, exist_ok=True)
    EVENT_LOG.touch(exist_ok=True)
    ev = {"ts": datetime.now(timezone.utc).isoformat(), **ev}
    with EVENT_LOG.open("a") as fp:
        fp.write(json.dumps(ev, ensure_ascii=False) + "\n")

# Minimal demo ATC dictionary (extend as needed)
ATC = {
    "ASS": "B01AC06", "Aspirin": "B01AC06", "Acetylsalicylsäure": "B01AC06",
    "Ibuprofen": "M01AE01",
    "Metformin": "A10BA02",
    "Ramipril": "C09AA05",
    "Simvastatin": "C10AA01",
    "Amoxicillin": "J01CA04", "Amoxicillin/Clavulansäure": "J01CR02",
    "Clarithromycin": "J01FA09",
    "Phenprocoumon": "B01AA04",
    "Spironolacton": "C03DA01",
}

# Demo interaction rules (replace with real rules JSON at MED_RULES_PATH for production)
DEMO_RULES = [
    # tuples of ATC or name patterns with a message
    (["J01FA09","C10AA01"], "Clarithromycin + Simvastatin: risk of myopathy/rhabdomyolysis"),
    (["C09AA05","C03DA01"], "ACE inhibitor + Spironolactone: risk of hyperkalaemia"),
    (["M01AE01","C09AA05"], "NSAID (Ibuprofen) + ACE inhibitor (Ramipril): risk of renal impairment"),
    (["B01AA04","B01AC06"], "Phenprocoumon + ASS: increased bleeding risk"),
]

# Very conservative allergy heuristics (demo)
ALLERGY_CLASS_MAP = {
    "penicillin": ["J01C", "J01CA", "J01CR", "J01CF"],
    "ibuprofen": ["M01AE"],
    "aspirin": ["B01AC06","N02BA"],
}

def _load_rules() -> List[Tuple[List[str], str]]:
    path = Path(CONFIG["MED_RULES_PATH"])
    if path.exists():
        try:
            js = json.loads(path.read_text())
            # expect [{"combo":["ATC1","ATC2"], "msg":"..."}]
            rules = []
            for r in js:
                combo = r.get("combo") or r.get("codes") or []
                msg = r.get("msg") or r.get("message") or "Interaction"
                if isinstance(combo, list) and combo:
                    rules.append((combo, msg))
            if rules:
                return rules
        except Exception:
            pass
    return DEMO_RULES

def _load_allergies() -> List[str]:
    path = Path(CONFIG["ALLERGIES_PATH"])
    if path.exists():
        try:
            js = json.loads(path.read_text())
            if isinstance(js, dict) and "allergies" in js and isinstance(js["allergies"], list):
                return js["allergies"]
        except Exception:
            pass
    return []

FREQ_MAP = {
    "morgens": "morning", "mittags": "noon", "abends": "evening", "nachts": "night",
    "1-0-0": "morning", "0-1-0": "noon", "0-0-1": "evening", "1-1-1": "tid",
    "1-0-1": "bid", "1-1-0": "bid", "2-0-0": "morning*2"
}

@dataclass
class MedEntry:
    name: str
    atc: Optional[str]
    strength: Optional[str]
    route: Optional[str]
    freq: Optional[str]
    prn: bool

def _normalize_name(name: str) -> str:
    name = name.strip()
    # strip common forms
    name = re.sub(r"\b(retard|sr|tbl|tab|tabl|tabletten|kapseln|lösung|saf t|sft)\b", "", name, flags=re.I)
    return re.sub(r"\s+", " ", name).strip()

def _guess_atc(name: str) -> Optional[str]:
    key = name
    if key in ATC: return ATC[key]
    # try title-case exact
    key = name.split()[0].capitalize()
    return ATC.get(key)

def parse_bmp_text(txt: str) -> List[MedEntry]:
    meds: List[MedEntry] = []
    # Split lines and parse patterns like "Metformin 1000 mg 1-0-1 p.o." or "Ibuprofen 400 mg bei Bedarf"
    for raw in txt.splitlines():
        line = raw.strip()
        if not line or line.startswith("#"): continue
        # Extract med name (words until first number or PRN marker)
        m = re.match(r"(?P<name>[A-Za-zÄÖÜäöüß/ \-]+)\s+(?P<rest>.*)", line)
        if not m: 
            continue
        name = _normalize_name(m.group("name"))
        rest = m.group("rest")

        # strength + unit
        m2 = re.search(r"(?P<dose>\d+(?:[.,]\d+)?)\s*(?P<unit>mg|g|ml|IE|µg)", rest, flags=re.I)
        strength = None
        if m2:
            strength = f"{m2.group('dose').replace(',', '.')} {m2.group('unit').upper()}"

        # route
        m3 = re.search(r"\b(p\.o\.|i\.v\.|s\.c\.|i\.m\.|p\.r\.)\b", rest, flags=re.I)
        route = m3.group(0).lower() if m3 else None

        # frequency
        m4 = re.search(r"\b(\d-\d-\d)\b", rest)
        freq = None
        if m4:
            freq = FREQ_MAP.get(m4.group(1), m4.group(1))
        else:
            # words
            for de, en in FREQ_MAP.items():
                if re.search(rf"\b{re}\b", rest, flags=re.I):
                    freq = en; break

        prn = bool(re.search(r"\b(prn|bedarf|bei bedarf)\b", rest, flags=re.I))
        atc = _guess_atc(name)

        meds.append(MedEntry(name=name, atc=atc, strength=strength, route=route, freq=freq, prn=prn))
    return meds

def check_interactions(meds: List[MedEntry]) -> List[Dict[str, Any]]:
    rules = _load_rules()
    codes = set([m.atc for m in meds if m.atc])
    names = set([m.name for m in meds])
    findings = []
    for combo, msg in rules:
        # treat combo entries as ATC prefixes or exact names/ATC
        present = 0
        for token in combo:
            token = str(token)
            if token in codes or token in names:
                present += 1
                continue
            # ATC prefix match (e.g., J01C* class)
            if any(c and c.startswith(token) for c in codes):
                present += 1
        if present == len(combo):
            findings.append({"type":"med_interaction_warning","msg": msg, "combo": combo})
    return findings

def check_allergies(meds: List[MedEntry], allergies: List[str]) -> List[Dict[str, Any]]:
    finds = []
    norm_all = [a.lower() for a in allergies]
    for m in meds:
        # direct string hit
        if any(a in m.name.lower() for a in norm_all):
            finds.append({"type":"med_allergy_warning","msg": f"Allergy matches {m.name}", "med": m.name})
        # class proximity
        if m.atc:
            for a in norm_all:
                classes = ALLERGY_CLASS_MAP.get(a)
                if not classes: continue
                if any(m.atc.startswith(pref) for pref in classes):
                    finds.append({"type":"med_allergy_warning","msg": f"Allergy '{a}' overlaps ATC class of {m.name} ({m.atc})", "med": m.name})
    return finds

def emit_med_events(patient_id: str, meds: List[MedEntry]):
    # summary
    _append_event({"type":"med_plan_import","patient_id": patient_id, "n": len(meds)})
    for m in meds:
        _append_event({"type":"med_entry","patient_id": patient_id, "name": m.name, "atc": m.atc,
                       "strength": m.strength, "route": m.route, "freq": m.freq, "prn": m.prn})

def process_bmp_text(patient_id: str, txt: str):
    meds = parse_bmp_text(txt)
    emit_med_events(patient_id, meds)
    warnings = []
    warnings += check_interactions(meds)
    warnings += check_allergies(meds, _load_allergies())
    for w in warnings:
        _append_event({**w, "patient_id": patient_id})
    return meds, warnings

DEMO_BMP = """\
# Demo Bundeseinheitlicher Medikationsplan (free-text)
Metformin 1000 mg 1-0-1 p.o.
Ramipril 5 mg 1-0-0 p.o.
Ibuprofen 400 mg bei Bedarf p.o.
Simvastatin 20 mg 0-0-1 p.o.
Clarithromycin 500 mg 1-0-1 p.o. (für 5 Tage)
Phenprocoumon 3 mg 1-0-0 p.o.
"""

if CONFIG.get("RUN_MEDS"):
    # For non-UI runs, process DEMO_BMP for patient SYN-DEMO-1
    meds, warns = process_bmp_text("SYN-DEMO-1", DEMO_BMP)
    print(f"[MEDS] Parsed {len(meds)} meds; warnings: {len(warns)}")
else:
    print("Deferred… set CONFIG['RUN_MEDS']=True for meds demo.")


In [ ]:

# UI: Medication Plan Import (Paste or OCR), interaction + allergy checks
if CONFIG.get("RUN_UI") and CONFIG.get("RUN_MEDS"):
    try:
        import ipywidgets as W
        import pandas as pd
        import os
        
        pid = W.Text(description="Patient ID", value="SYN-DEMO-1")
        path = W.Text(description="Scan path", placeholder="/mnt/data/scan.pdf or .png")
        ocr_btn = W.Button(description="Run OCR")
        txt = W.Textarea(value=DEMO_BMP, description="Plan text", layout=W.Layout(width="100%", height="160px"))
        parse_btn = W.Button(description="Parse & Check", button_style="primary")
        out = W.Output()

        def _ocr_run(_):
            with out:
                out.clear_output()
                try:
                    p = Path(path.value.strip())
                    if not p.exists():
                        print("File not found:", p)
                        return
                    # Best-effort OCR: pdf2image + pytesseract if present
                    txt_val = None
                    try:
                        import pytesseract
                        from PIL import Image
                        if p.suffix.lower()==".pdf":
                            from pdf2image import convert_from_path
                            pages = convert_from_path(str(p))
                            text_parts = []
                            for page in pages:
                                text_parts.append(pytesseract.image_to_string(page, lang="deu"))
                            txt_val = "\n".join(text_parts)
                        else:
                            img = Image.open(str(p))
                            txt_val = pytesseract.image_to_string(img, lang="deu")
                    except Exception as e:
                        print("OCR unavailable or failed:", e)
                    if txt_val:
                        txt.value = txt_val
                        print("OCR complete → text area filled.")
                except Exception as e:
                    print("OCR error:", e)

        def _parse_run(_):
            with out:
                out.clear_output()
                meds, warns = process_bmp_text(pid.value.strip() or "DEMO", txt.value)
                if meds:
                    df = pd.DataFrame([m.__dict__ for m in meds])
                    display(df.style.hide(axis='index'))
                if warns:
                    print("\nWarnings:")
                    for w in warns:
                        print(" -", w["type"], ":", w.get("msg"))
                print("\nEvents appended to", CONFIG["EVENT_LOG_PATH"])

        ocr_btn.on_click(_ocr_run); parse_btn.on_click(_parse_run)
        display(W.VBox([W.HBox([pid]), W.HBox([path, ocr_btn]), txt, parse_btn, out]))
    except Exception as e:
        print("UI unavailable:", e)


In [ ]:

# Conference Demo: one-click run (synthetic ML + meds import + ICU ETA)
if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W, json, pandas as pd, importlib.util
        run_btn = W.Button(description="Run Full Demo", button_style="success")
        out = W.Output()

        def _run(_):
            with out:
                out.clear_output()
                # 1) Synthetic ML pipeline (if configured)
                try:
                    CONFIG["RUN_SYNTH"] = True
                    print("[1/3] Running synthetic ML pipeline…")
                    # reuse the run_synth_pipeline if defined
                    try:
                        res = run_synth_pipeline(save_csv=False)
                    except NameError:
                        print("run_synth_pipeline not loaded in this kernel; skip")
                    else:
                        print("  →", res)
                except Exception as e:
                    print("Synthetic pipeline error:", e)

                # 2) ICU ETAs (if ICU panel code is present)
                try:
                    from datetime import datetime, timezone
                    # create a tiny scenario: set one unit to open in 30 min
                    state = {"timestamp": datetime.now(timezone.utc).isoformat(),
                             "units": [{"name":"1A Neurochirurgische Intensivstation","capacity":12,"occupied":12,"discharge_eta_minutes":[30,120]},
                                       {"name":"H2b Intensivstation Gefäß- und Herzmedizin","capacity":8,"occupied":7,"discharge_eta_minutes":[]} ]}
                    Path("/mnt/data/icu_status.json").write_text(json.dumps(state, ensure_ascii=False, indent=2))
                    print("[2/3] ICU sample state saved → /mnt/data/icu_status.json")
                except Exception as e:
                    print("ICU demo setup error:", e)

                # 3) Meds import demo
                try:
                    print("[3/3] Parsing demo BMP for patient SYN-DEMO-1…")
                    meds, warns = process_bmp_text("SYN-DEMO-1", DEMO_BMP)
                    print(f"  → {len(meds)} meds, {len(warns)} warnings; events appended.")
                except Exception as e:
                    print("Meds demo error:", e)

                # Tail event log
                try:
                    p = Path(CONFIG["EVENT_LOG_PATH"])
                    if p.exists():
                        lines = p.read_text().strip().splitlines()[-10:]
                        print("\nEvent log tail:")
                        for ln in lines:
                            print(" ", ln[:200])
                except Exception as e:
                    print("Log tail error:", e)

        run_btn.on_click(_run)
        display(W.VBox([run_btn, out]))
    except Exception as e:
        print("Demo UI unavailable:", e)
else:
    print("Demo UI deferred… set CONFIG['RUN_UI']=True.")


In [ ]:

# === Medication Plan Import + Checks (inline, no sidecars) ===
from __future__ import annotations
from typing import Dict, Any, List, Optional, Tuple
from pathlib import Path
from datetime import datetime, timezone
import json, re

EVENT_LOG = Path(CONFIG["EVENT_LOG_PATH"])

def _append_event(ev: Dict[str, Any]):
    EVENT_LOG.parent.mkdir(parents=True, exist_ok=True)
    EVENT_LOG.touch(exist_ok=True)
    ev = {"ts": datetime.now(timezone.utc).isoformat(), **ev}
    with EVENT_LOG.open("a") as fp:
        fp.write(json.dumps(ev, ensure_ascii=False) + "\n")

# Minimal ATC/name map for demo (extendable)
_ATC_MAP = {
    "amoxicillin": {"atc": "J01CA04", "class": "penicillin"},
    "ibuprofen": {"atc": "M01AE01", "class": "nsaid"},
    "warfarin": {"atc": "B01AA03", "class": "coumarin"},
    "ramipril": {"atc": "C09AA05", "class": "ace"},
    "spironolacton": {"atc": "C03DA01", "class": "aldosterone_antagonist"},
    "metoprolol": {"atc": "C07AB02", "class": "beta_blocker"},
    "simvastatin": {"atc": "C10AA01", "class": "statin"},
    "clarithromycin": {"atc": "J01FA09", "class": "macrolide"},
    "azithromycin": {"atc": "J01FA10", "class": "macrolide"},
}

# Demo interaction rules (replace via MED_RULES_PATH for your own rules)
_DEMO_RULES = [
    {"a": "ibuprofen", "b": "warfarin", "severity": "major", "message": "Bleeding risk (NSAID + warfarin)"},
    {"a": "ramipril", "b": "spironolacton", "severity": "moderate", "message": "Hyperkalemia risk (ACE + aldosterone antagonist)"},
    {"a": "amoxicillin", "b": "warfarin", "severity": "moderate", "message": "Antibiotic may potentiate warfarin effect"},
    {"a": "clarithromycin", "b": "simvastatin", "severity": "major", "message": "Rhabdomyolysis risk (CYP3A4 inhibition)"},
]

_ROUTES = ["p.o.", "po", "i.v.", "iv", "i.m.", "im", "s.c.", "sc", "inhalativ", "topisch", "nasal", "otic", "ophthalmic"]
_FREQ_WORDS = ["morgens", "mittags", "abends", "nachts"]
_FREQ_PAT = re.compile(r"\b(\d+)[-/.](\d+)[-/.](\d+)\b")
_DOSE_PAT = re.compile(r"(\d+(?:[.,]\d+)?)\s*(mg|g|mcg|µg|ml|IE|Einheiten)\b", flags=re.I)

def _normalize_name(s: str) -> str:
    s = s.strip().lower()
    s = s.replace("ä","ae").replace("ö","oe").replace("ü","ue").replace("ß","ss")
    return re.sub(r"[^a-z0-9]+", " ", s).strip()

def _map_to_atc(name_norm: str) -> Dict[str, Any]:
    for key, meta in _ATC_MAP.items():
        if key in name_norm:
            return {"name_norm": key, **meta}
    return {"name_norm": name_norm, "atc": None, "class": None}

def parse_med_line(line: str) -> Optional[Dict[str, Any]]:
    raw = line.strip()
    if not raw or raw.startswith("#"):
        return None
    name = raw.split(",")[0].split("  ")[0]  # up to comma or double spaces
    name_norm = _normalize_name(name)
    dose_val, dose_unit = None, None
    m = _DOSE_PAT.search(raw)
    if m:
        dose_val = float(m.group(1).replace(",", "."))
        dose_unit = m.group(2).lower()
    freq = None
    m = _FREQ_PAT.search(raw)
    if m:
        freq = f"{m.group(1)}-{m.group(2)}-{m.group(3)}"
    else:
        words = [w for w in _FREQ_WORDS if w in raw.lower()]
        if words:
            freq = ",".join(words)
    route = None
    for r in _ROUTES:
        if r in raw.lower():
            route = r
            break
    prn = bool(re.search(r"\b(prn|bei bedarf)\b", raw, flags=re.I))
    atc_meta = _map_to_atc(name_norm)
    return {
        "raw": raw,
        "name": name.strip(),
        "name_norm": atc_meta["name_norm"],
        "atc": atc_meta["atc"],
        "class": atc_meta["class"],
        "dose_value": dose_val,
        "dose_unit": dose_unit,
        "frequency": freq,
        "route": route,
        "prn": prn,
    }

def parse_med_text(text: str) -> List[Dict[str, Any]]:
    meds = []
    for line in text.splitlines():
        rec = parse_med_line(line)
        if rec:
            meds.append(rec)
    return meds

def load_rules(path: str) -> List[Dict[str, Any]]:
    p = Path(path)
    if p.exists():
        try:
            js = json.loads(p.read_text())
            if isinstance(js, list):
                return js
        except Exception:
            pass
    return list(_DEMO_RULES)

def load_allergies(path: str) -> List[str]:
    p = Path(path)
    if p.exists():
        try:
            js = json.loads(p.read_text())
            if isinstance(js, dict) and "allergies" in js and isinstance(js["allergies"], list):
                return [str(x) for x in js["allergies"]]
        except Exception:
            pass
    return []

def check_interactions(meds: List[Dict[str, Any]], rules: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out = []
    names = [m["name_norm"] for m in meds]
    for r in rules:
        a, b = r["a"], r["b"]
        if (a in names and b in names) or (b in names and a in names):
            out.append({"type": "med_interaction_warning", **r})
    return out

def check_allergies(meds: List[Dict[str, Any]], allergy_terms: List[str]) -> List[Dict[str, Any]]:
    out = []
    terms = [t.lower() for t in allergy_terms]
    for m in meds:
        # direct name match
        if any(t in m["name"].lower() for t in terms):
            out.append({"type": "med_allergy_warning", "med": m["name"], "match": "name"})
            continue
        # conservative penicillin class heuristic
        if any("penicillin" in t for t in terms):
            if m["class"] == "penicillin" or m["name"].lower().endswith("cillin"):
                out.append({"type": "med_allergy_warning", "med": m["name"], "match": "class_penicillin"})
    return out

def ocr_file(path: str) -> Optional[str]:
    p = Path(path)
    if not p.exists():
        return None
    try:
        from PIL import Image
        import pytesseract
        if p.suffix.lower() in [".png",".jpg",".jpeg",".tif",".tiff"]:
            return pytesseract.image_to_string(Image.open(p))
        if p.suffix.lower() == ".pdf":
            try:
                from pdf2image import convert_from_path
                pages = convert_from_path(str(p))
                text = ""
                for img in pages:
                    text += pytesseract.image_to_string(img) + "\n"
                return text
            except Exception:
                return None
    except Exception:
        return None
    return None

def emit_meds_events(patient_id: str, meds: List[Dict[str, Any]], warnings: List[Dict[str, Any]]):
    _append_event({"type": "med_plan_import", "patient_id": patient_id, "n_meds": len(meds)})
    for m in meds:
        _append_event({"type": "med_entry", "patient_id": patient_id, **m})
    for w in warnings:
        _append_event({**w, "patient_id": patient_id})

if CONFIG.get("RUN_MEDS"):
    print("Medication import/checks enabled. Use the UI panel (if RUN_UI=True) or call the functions above.")
else:
    print("Deferred… set CONFIG['RUN_MEDS']=True to enable medication import & checks.")

# Optional UI
if CONFIG.get("RUN_UI") and CONFIG.get("RUN_MEDS"):
    try:
        import ipywidgets as W
        import pandas as pd
        pid = W.Text(description="Patient ID", placeholder="e.g., UKE-12345")
        path = W.Text(description="Scan path", placeholder="/mnt/data/scan.pdf (optional)")
        ocr_btn = W.Button(description="Run OCR")
        ta = W.Textarea(description="Plan text", layout=W.Layout(width="100%", height="180px"))
        parse_btn = W.Button(description="Parse & Check", button_style="primary")
        out = W.Output()

        def on_ocr(_):
            with out:
                out.clear_output()
                txt = ocr_file(path.value.strip())
                if txt:
                    ta.value = txt
                    print("OCR complete.")
                else:
                    print("OCR unavailable or failed. Paste the text instead.")

        def on_parse(_):
            with out:
                out.clear_output()
                text = ta.value.strip()
                if not text:
                    print("No text provided."); return
                meds = parse_med_text(text)
                rules = load_rules(CONFIG["MED_RULES_PATH"])
                allergies = load_allergies(CONFIG["ALLERGIES_PATH"])
                warnings = check_interactions(meds, rules) + check_allergies(meds, allergies)
                emit_meds_events(pid.value or "DEMO", meds, warnings)
                if meds:
                    display(pd.DataFrame(meds))
                else:
                    print("No medications parsed.")
                if warnings:
                    print("\nWarnings:")
                    display(pd.DataFrame(warnings))
                else:
                    print("\nNo interaction/allergy warnings.")

        ocr_btn.on_click(on_ocr)
        parse_btn.on_click(on_parse)
        display(W.VBox([pid, path, W.HBox([ocr_btn, parse_btn]), ta, out]))
    except Exception as e:
        print("Meds UI unavailable:", e)


In [ ]:

# === Conference Demo: one-click walkthrough ===
from pathlib import Path
from datetime import datetime, timezone
import json

def _tail_events(n=12):
    p = Path(CONFIG["EVENT_LOG_PATH"])
    if not p.exists():
        return []
    lines = p.read_text().splitlines()
    return [json.loads(x) for x in lines[-n:]] if lines else []

if CONFIG.get("RUN_UI"):
    try:
        import ipywidgets as W, pandas as pd

        btn = W.Button(description="Conference Demo → Run Full Demo", button_style="success")
        out = W.Output()

        def run_demo(_):
            with out:
                out.clear_output()
                print("Running conference demo…")

                # Step 1: Synthetic ML pipeline (if enabled)
                if CONFIG.get("RUN_SYNTH"):
                    try:
                        res = run_synth_pipeline(save_csv=False)
                        print("[SYNTH]", res)
                    except Exception as e:
                        print("[SYNTH ERROR]", e)
                else:
                    print("[SYNTH] Skipped (set RUN_SYNTH=True to include)")

                # Step 2: ICU snapshot → next bed in 30 min on one unit
                icu = {
                    "timestamp": datetime.now(timezone.utc).isoformat(),
                    "units": [
                        {"name":"1C Interdisziplinäre Intensivstation","capacity":12,"occupied":12,"discharge_eta_minutes":[30,120,240]},
                        {"name":"1G Internistische Intensivstation","capacity":12,"occupied":11,"discharge_eta_minutes":[60]}
                    ]
                }
                Path("/mnt/data/icu_status.json").write_text(json.dumps(icu, ensure_ascii=False, indent=2))
                print("[ICU] Snapshot saved → next bed on 1C in 30 min")

                # Step 3: Medication plan demo (paste-mode)
                demo_text = """\
Amoxicillin 500 mg 1-0-1 p.o.
Ibuprofen 400 mg 1-1-1 p.o. bei Bedarf
Warfarin 5 mg 1-0-0 p.o.
Ramipril 5 mg 1-0-0 p.o.
Spironolacton 25 mg 0-0-1 p.o.
"""
                meds = parse_med_text(demo_text)
                rules = load_rules(CONFIG["MED_RULES_PATH"])
                allergies = ["Penicillin"]  # demo allergy to trigger warning with Amoxicillin
                warnings = check_interactions(meds, rules) + check_allergies(meds, allergies)
                emit_meds_events("DEMO-123", meds, warnings)
                print("[MEDS] Parsed", len(meds), "entries; warnings:", len(warnings))

                # Show event log tail
                tail = _tail_events(12)
                if tail:
                    display(pd.DataFrame(tail))
                else:
                    print("No events yet.")

        btn.on_click(run_demo)
        display(W.VBox([btn, out]))
    except Exception as e:
        print("Demo UI unavailable:", e)
else:
    print("Demo UI deferred… set CONFIG['RUN_UI']=True.")
